In [ ]:
import pandas as pd
import numpy as np
import json
import os
import math
import calendar
from datetime import datetime, date

# =============================================================
#  Smart APS V9  —  5-Day Target Inventory System
#  + Runner Priority Enforcement
#  + Corrected Terminal Constraint (inventory-based)
#  + Fixed Machine Constraint  ← NEW
#
#  Fixed Machine logic:
#  ─────────────────────────────────────────────────────────────
#  Sheet VT_Fixed in compatibility_matrix.xlsx
#  Columns: Machine | Part_1 | Part_2
#
#  Both Part_1 and Part_2 listed on a row prefer that machine
#  as their first assignment choice.
#
#  Rule: PREFERRED FIRST (Option B)
#    1. If fixed machine has effective capacity ≥ MIN_RUN_HOURS
#       after changeover → assign there, skip normal ranking.
#    2. If fixed machine has zero/insufficient capacity →
#       fall back to normal rank_machines() over all compatible
#       machines from VT_Matrix.
#
#  Fixed parts are all Runners.
#  Fixed preference OVERRIDES Runner Lock.
#  Enforcer does NOT apply fixed preference.
# =============================================================

# =============================================================
# SECTION 1 — DAILY SETTINGS
# =============================================================

PLANNING_DATE = date(2026, 4, 9)
INDENT_MONTH  = date(2026, 4, 1)

# =============================================================
# SECTION 2 — PARAMETERS
# =============================================================

AVAILABLE_HOURS      = 22
MIN_RUN_HOURS        = 4
MACHINE_STATE_FILE   = "machine_state.json"

MIN_DAILY_INDENT     = 150
MIN_INDENT_HOURS     = 4.0

SAFETY_DAYS  = 3
TARGET_DAYS  = 5

OPD_SCENARIO_0 = 1.5
OPD_SCENARIO_1 = 3.0
OPD_SCENARIO_2 = 4.0
OPD_SCENARIO_3 = 5.0

W_URGENCY  = 0.55
W_CATEGORY = 0.25
W_INDENT   = 0.20

UTIL_TARGET_PCT = 90.0

COLOR_PURGE_HRS = 10 / 60.0   # 10 minutes in hours

# =============================================================
# SECTION 3 — FILE PATHS
# =============================================================

book_path       = "C:/Users/Ex0164/Book1.xlsx"
matrix_path     = "C:/Users/Ex0164/Important codes/compatibility_matrix.xlsx"
changeover_path = "D:/Tushar/TOOL FIX/Unique_Machines_vt.xlsx"
terminal_path   = "C:/Users/Ex0164/Important codes/terminals and raw marterial - vt.xlsx"
output_path     = f"Smart_APS_V9_Plan_{PLANNING_DATE.strftime('%Y%m%d')} v22 fixed machine.xlsx"

# =============================================================
# SECTION 4 — WORKING DAYS
# =============================================================

def compute_working_days(ref_date):
    year  = ref_date.year
    month = ref_date.month
    total = calendar.monthrange(year, month)[1]
    sundays = sum(
        1 for d in range(1, total + 1)
        if date(year, month, d).weekday() == 6
    )
    return total - sundays, total, sundays

WORKING_DAYS, TOTAL_DAYS, SUNDAY_COUNT = compute_working_days(INDENT_MONTH)

print(f"\n{'='*65}")
print(f"  Smart APS V9  —  Fixed Machine + Runner Priority")
print(f"  Planning date : {PLANNING_DATE}")
print(f"  Indent month  : {INDENT_MONTH.strftime('%B %Y')}")
print(f"  Working days  : {WORKING_DAYS}  ({TOTAL_DAYS} days − {SUNDAY_COUNT} Sundays)")
print(f"  Safety floor  : {SAFETY_DAYS} days  |  Target ceiling : {TARGET_DAYS} days")
print(f"{'='*65}\n")

# =============================================================
# SECTION 5 — LOAD DATA
# =============================================================

print("Loading data...")
vt_parts_raw         = pd.read_excel(book_path,       sheet_name="VT")
vt_matrix            = pd.read_excel(matrix_path,     sheet_name="VT_Matrix")
vt_co_raw            = pd.read_excel(changeover_path, sheet_name="VT_Changeover")
vt_machine_count_raw = pd.read_excel(matrix_path,     sheet_name="VT_Machine_Part_Count")

# ── Fixed machine sheet ───────────────────────────────────────
try:
    vt_fixed_raw = pd.read_excel(matrix_path, sheet_name="VT_Fixed")
    print(f"  VT_Fixed sheet loaded  ({len(vt_fixed_raw)} rows)")
except Exception as _fe:
    vt_fixed_raw = None
    print(f"  WARNING: VT_Fixed sheet not found ({_fe})")
    print(f"           Fixed machine constraint DISABLED.")

# ── Terminal sheet ────────────────────────────────────────────
try:
    vt_terminals_raw      = pd.read_excel(terminal_path, sheet_name="VT_Terminals")
    vt_terminal_avail_raw = pd.read_excel(terminal_path, sheet_name="VT_Terminal_Inventory")
    print(f"  Terminal data loaded from  : {terminal_path}")
except FileNotFoundError:
    vt_terminals_raw      = None
    vt_terminal_avail_raw = None
    print(f"  WARNING: terminal_path not found ({terminal_path})")
    print(f"           Terminal constraint DISABLED.")
except Exception as _te:
    vt_terminals_raw      = None
    vt_terminal_avail_raw = None
    print(f"  WARNING: Could not load terminal data ({_te})")
    print(f"           Terminal constraint DISABLED.")

# =============================================================
# SECTION 6 — PARSE VT SHEET
# =============================================================

def find_col(df, name, sheet):
    match = next(
        (c for c in df.columns if str(c).strip().lower() == name.lower()), None
    )
    if match is None:
        raise ValueError(
            f"Column '{name}' not found in sheet '{sheet}'.\n"
            f"Available: {list(df.columns)}"
        )
    return match

vt_col_part      = find_col(vt_parts_raw, "Part",       "VT")
vt_col_cycletime = find_col(vt_parts_raw, "Cycle time", "VT")
vt_col_cavity    = find_col(vt_parts_raw, "Cavity",     "VT")
vt_col_inventory = find_col(vt_parts_raw, "Inventory",  "VT")
vt_col_indent    = find_col(vt_parts_raw, "Indent",     "VT")
vt_col_tools     = find_col(vt_parts_raw, "Tools",      "VT")
vt_col_color     = find_col(vt_parts_raw, "Color",      "VT")

data = vt_parts_raw[
    vt_parts_raw[vt_col_part].notna() &
    (vt_parts_raw[vt_col_part].astype(str).str.strip() != "")
].copy()
data = data.drop_duplicates(subset=vt_col_part).copy()
data["Material"] = data[vt_col_part].astype(str).str.strip()

data["_ct"] = pd.to_numeric(data[vt_col_cycletime], errors="coerce").replace(0, np.nan)
data["Rate"] = 3600 / data["_ct"]

data_valid     = data[data["Rate"].notna()].copy()
data_zero_rate = data[data["Rate"].isna()].copy()

print(f"  VT parts in sheet       : {len(data)}")
print(f"  Parts with valid rate   : {len(data_valid)}")

# =============================================================
# SECTION 7 — LOOKUP DICTIONARIES
# =============================================================

def safe_dict(df, key_col, val_col, default=0.0):
    return {
        str(k).strip(): (default if pd.isna(v) else float(v))
        for k, v in zip(df[key_col], df[val_col])
        if pd.notna(k) and str(k).strip() != ""
    }

inventory      = safe_dict(data,       "Material", vt_col_inventory)
rate           = safe_dict(data_valid, "Material", "Rate")
indent_monthly = safe_dict(data,       "Material", vt_col_indent)

tools_available = {}
part_color      = {}

for _, row in data.iterrows():
    p = str(row["Material"]).strip()
    v = row[vt_col_tools]
    tools_available[p] = (
        max(1, int(float(v)))
        if pd.notna(v) and str(v).strip() != ""
        else 1
    )
    c = row[vt_col_color]
    part_color[p] = (
        str(c).strip().upper()
        if pd.notna(c) and str(c).strip() not in ("", "nan")
        else "UNKNOWN"
    )

color_groups = {}
for p, c in part_color.items():
    color_groups.setdefault(c, []).append(p)
print(f"  Distinct colours        : {len(color_groups)}")
for col, pts in sorted(color_groups.items()):
    print(f"    {col:<20} → {len(pts)} part(s)")

indent_daily = {
    p: round(qty / WORKING_DAYS, 4)
    for p, qty in indent_monthly.items()
}

today_target_qty = {
    p: max(0.0, indent_daily.get(p, 0.0) - inventory.get(p, 0.0))
    for p in indent_monthly
}

# =============================================================
# SECTION 7A — FIXED MACHINE CONSTRAINT  ← NEW
# =============================================================
#
# Builds two dicts:
#   part_fixed_machine : {part → machine_name}
#     The machine a part prefers to run on.
#     Built from VT_Fixed sheet (Machine | Part_1 | Part_2).
#
#   machine_fixed_parts : {machine → [part, ...]}
#     Which parts are fixed to a machine (for display/audit).
#
# A part may appear in at most ONE row (one fixed machine).
# If a part appears in both Part_1 and Part_2 columns across
# different rows, the first occurrence wins (log a warning).
# =============================================================

def build_fixed_machine_dicts(df):
    """
    Reads VT_Fixed sheet.
    Columns: Machine | Part_1 | Part_2  (exact names, case-insensitive)
    Returns:
        part_fixed_machine  : {part_str → machine_str}
        machine_fixed_parts : {machine_str → [part_str, ...]}
    """
    pfm  = {}   # part  → machine
    mfp  = {}   # machine → [parts]

    if df is None or df.empty:
        return pfm, mfp

    # Find column names flexibly
    machine_col = next(
        (c for c in df.columns if str(c).strip().lower() == "machine"), None
    )
    if machine_col is None:
        print("  WARNING: VT_Fixed sheet has no 'Machine' column — fixed constraint disabled")
        return pfm, mfp

    # Collect all part columns (everything that is not 'machine')
    part_cols = [
        c for c in df.columns
        if str(c).strip().lower() != "machine"
    ]

    if not part_cols:
        print("  WARNING: VT_Fixed sheet has no part columns — fixed constraint disabled")
        return pfm, mfp

    for _, row in df.iterrows():
        machine = row[machine_col]
        if pd.isna(machine) or str(machine).strip() == "":
            continue
        m = str(machine).strip()

        for col in part_cols:
            val = row[col]
            if pd.isna(val) or str(val).strip() in ("", "nan"):
                continue
            p = str(val).strip()

            if p in pfm:
                print(f"  WARNING: Part '{p}' appears in VT_Fixed more than once "
                      f"— keeping first assignment ({pfm[p]}), ignoring {m}")
                continue

            pfm[p] = m
            mfp.setdefault(m, []).append(p)

    return pfm, mfp


part_fixed_machine, machine_fixed_parts = build_fixed_machine_dicts(vt_fixed_raw)

print(f"  Fixed machine mappings  : {len(part_fixed_machine)} parts")
if part_fixed_machine:
    for m, parts in sorted(machine_fixed_parts.items()):
        print(f"    {m:<25} ← {', '.join(parts)}")

# =============================================================
# SECTION 7B — TERMINAL CONSTRAINT
# =============================================================

def _build_part_terminals(df):
    result = {}
    if df is None or df.empty:
        return result
    part_col = next(
        (c for c in df.columns
         if str(c).strip().lower() in ("part", "material")), None
    )
    if part_col is None:
        print("  WARNING: VT_Terminals sheet has no 'Part'/'Material' column "
              "— terminal constraint disabled")
        return result
    terminal_cols = [
        c for c in df.columns
        if str(c).strip().lower() not in ("part", "material")
    ]
    for _, row in df.iterrows():
        part = row[part_col]
        if pd.isna(part) or str(part).strip() == "":
            continue
        p = str(part).strip()
        terminals = []
        for col in terminal_cols:
            val = row[col]
            if pd.notna(val) and str(val).strip() not in ("", "nan"):
                terminals.append(str(val).strip().upper())
        if terminals:
            result[p] = terminals
    return result


def _build_terminal_status(df):
    result = {}
    if df is None or df.empty:
        return result
    term_col = next(
        (c for c in df.columns if str(c).strip().lower() == "terminal"), None
    )
    inv_col = next(
        (c for c in df.columns if str(c).strip().lower() == "inventory"), None
    )
    if term_col is None or inv_col is None:
        print("  WARNING: VT_Terminal_Inventory missing 'Terminal' or "
              "'Inventory' column — terminal constraint disabled")
        return result
    for _, row in df.iterrows():
        t = row[term_col]
        v = row[inv_col]
        if pd.isna(t) or str(t).strip() == "":
            continue
        key = str(t).strip().upper()
        try:
            qty = float(v) if pd.notna(v) else 0.0
        except (ValueError, TypeError):
            qty = 0.0
        result[key] = qty
    return result


part_terminals  = _build_part_terminals(vt_terminals_raw)
terminal_status = _build_terminal_status(vt_terminal_avail_raw)

if terminal_status:
    zero_count  = sum(1 for v in terminal_status.values() if v <= 0)
    avail_count = len(terminal_status) - zero_count
    print(f"  Terminals loaded : {len(terminal_status)} total  |  "
          f"{avail_count} with stock  |  {zero_count} at ZERO inventory")
    if zero_count:
        zero_list = [
            f"{t}(inv={v:.0f})"
            for t, v in terminal_status.items() if v <= 0
        ]
        print(f"  Zero-stock terminals : {', '.join(sorted(zero_list))}")
        blocked_parts = [
            p for p, terms in part_terminals.items()
            if any(terminal_status.get(t, 0) <= 0 for t in terms)
        ]
        print(f"  Parts blocked by zero terminal inventory: {len(blocked_parts)}")
else:
    print(f"  Terminals        : no data loaded — constraint inactive")


def terminal_blocked(part):
    required = part_terminals.get(part, [])
    if not required:
        return False, ""
    zero_inv = [t for t in required if terminal_status.get(t, 0) <= 0]
    if zero_inv:
        details = ", ".join(
            f"{t}(inv={terminal_status.get(t, 0):.0f})"
            for t in sorted(zero_inv)
        )
        return True, (
            f"Terminal(s) with zero inventory: {details}  "
            f"(requires: {', '.join(required)})"
        )
    return False, ""

# =============================================================
# SECTION 7C — SKIP RULES
# =============================================================

def should_skip(part):
    daily   = indent_daily.get(part, 0.0)
    monthly = indent_monthly.get(part, 0.0)
    r       = rate.get(part, 1.0)

    if daily <= MIN_DAILY_INDENT:
        return True, f"Daily indent {daily:.2f} ≤ {MIN_DAILY_INDENT} threshold"

    indent_hrs = monthly / r if r > 0 else 0.0
    if indent_hrs <= MIN_INDENT_HOURS:
        return True, f"Whole monthly indent = {indent_hrs:.2f}h ≤ {MIN_INDENT_HOURS}h threshold"

    inv = inventory.get(part, 0.0)
    if daily > 0 and inv >= TARGET_DAYS * daily:
        return True, (
            f"Inventory ({inv:.0f}) ≥ {TARGET_DAYS}-day target "
            f"({TARGET_DAYS * daily:.0f} pcs) — at ceiling, skip today"
        )

    t_blocked, t_reason = terminal_blocked(part)
    if t_blocked:
        return True, t_reason

    return False, ""

# =============================================================
# SECTION 7D — CHANGEOVER TIMES
# =============================================================

def build_changeover_dict(co_df):
    co_dict = {}
    for _, row in co_df.iterrows():
        machine = str(row["Unique Machines"]).strip()
        minutes = row["Changeover time"]
        if pd.notna(minutes) and machine:
            co_dict[machine] = float(minutes) / 60.0
    return co_dict

vt_changeover          = build_changeover_dict(vt_co_raw)
DEFAULT_CHANGEOVER_HRS = 40 / 60.0

# =============================================================
# SECTION 7E — MACHINE PART COUNT
# =============================================================

def build_machine_part_count(df):
    mpc = {}
    machine_col = next(
        (c for c in df.columns if str(c).strip().lower() == "machine"), None
    )
    count_col = next(
        (c for c in df.columns if str(c).strip().lower() == "part_count"), None
    )
    if machine_col is None or count_col is None:
        print("  WARNING: VT_Machine_Part_Count sheet missing 'Machine' or 'Part_Count' column")
        return {}
    for _, row in df.iterrows():
        m = str(row[machine_col]).strip()
        v = row[count_col]
        if m and pd.notna(v):
            try:
                mpc[m] = int(float(v))
            except (ValueError, TypeError):
                pass
    return mpc

machine_part_count = build_machine_part_count(vt_machine_count_raw)
max_part_count     = max(machine_part_count.values(), default=1) or 1
print(f"  Machine part counts loaded: {len(machine_part_count)} machines")

# =============================================================
# SECTION 7F — PART CATEGORY
# =============================================================

def build_category(df):
    cat      = {}
    part_col = next(
        (c for c in df.columns if str(c).strip().lower() == "part"), None
    )
    cat_col  = next(
        (c for c in df.columns if str(c).strip().lower() == "category"), None
    )
    if part_col is None:
        return cat
    for _, row in df.iterrows():
        part = row[part_col]
        if pd.isna(part) or str(part).strip() == "":
            continue
        val = "Stranger"
        if cat_col and pd.notna(row[cat_col]):
            val = str(row[cat_col]).strip().capitalize()
            if val not in ("Runner", "Repeater", "Stranger"):
                val = "Stranger"
        cat[str(part).strip()] = val
    return cat

part_category  = build_category(vt_parts_raw)
CATEGORY_SCORE = {"Runner": 100, "Repeater": 60, "Stranger": 20}

# =============================================================
# SECTION 8 — MACHINE STATE
# =============================================================

def load_machine_state():
    if os.path.exists(MACHINE_STATE_FILE):
        try:
            with open(MACHINE_STATE_FILE) as f:
                content = f.read().strip()
            if not content:
                print(f"  Machine state : file empty — treating as first run")
                os.remove(MACHINE_STATE_FILE)
                return {}
            state = json.loads(content)
            if not isinstance(state, dict):
                print(f"  Machine state : file corrupt — treating as first run")
                os.remove(MACHINE_STATE_FILE)
                return {}
            print(f"  Machine state loaded  ({len(state)} machines with history)")
            return state
        except Exception as e:
            print(f"  Machine state : error ({e}) — treating as first run")
            os.remove(MACHINE_STATE_FILE)
            return {}
    print(f"  Machine state : FIRST RUN — no changeover today")
    return {}

def save_machine_state(state):
    combined = {m: p for m, p in state.items() if p is not None}
    with open(MACHINE_STATE_FILE, "w") as f:
        json.dump(combined, f, indent=2)
    print(f"\n  Machine state saved → '{MACHINE_STATE_FILE}'")

machine_state = load_machine_state()

# =============================================================
# SECTION 9 — COMPATIBILITY MATRIX
# =============================================================

def build_compatibility(matrix):
    machines = matrix.columns[1:].tolist()
    compat   = {}
    for _, row in matrix.iterrows():
        part = row["Part"]
        for m in machines:
            if row[m] == 1:
                compat.setdefault(part, []).append(m)
    return compat, machines

vt_compat, vt_machines = build_compatibility(vt_matrix)

# =============================================================
# SECTION 10 — SCENARIO CLASSIFIER
# =============================================================

def classify_scenario(parts):
    coverage = []
    for p in parts:
        inv   = inventory.get(p, 0)
        daily = indent_daily.get(p, 0)
        skip, _ = should_skip(p)
        if skip or daily == 0:
            continue
        coverage.append(inv / daily)

    if not coverage:
        return 3, "SCENARIO 3 — No active parts today"

    n        = len(coverage)
    critical = sum(1 for c in coverage if c < 1)
    low      = sum(1 for c in coverage if c < SAFETY_DAYS)

    if critical == n:
        return 0, f"SCENARIO 0 — ALL {n} active parts critical (inv < 1 day)"
    elif critical > 0:
        return 1, f"SCENARIO 1 — {critical}/{n} parts critical  |  {n-critical} have buffer"
    elif low > 0:
        return 2, f"SCENARIO 2 — {low}/{n} parts below {SAFETY_DAYS}-day safety floor"
    else:
        return 3, f"SCENARIO 3 — All {n} parts healthy (≥{SAFETY_DAYS} days safety floor)"

# =============================================================
# SECTION 11 — OPD CAP
# =============================================================

def opd_cap(scenario_id):
    scenario_opd = {
        0: OPD_SCENARIO_0,
        1: OPD_SCENARIO_1,
        2: OPD_SCENARIO_2,
        3: OPD_SCENARIO_3,
    }.get(scenario_id, OPD_SCENARIO_2)
    return min(scenario_opd, TARGET_DAYS)

# =============================================================
# SECTION 12 — PRIORITY SCORING
# =============================================================

def compute_priority_scores(active_parts):
    rows = []
    for p in active_parts:
        inv      = inventory.get(p, 0)
        daily    = indent_daily.get(p, 0)
        cat      = part_category.get(p, "Stranger")
        days_cov = inv / daily if daily > 0 else 999.0
        gap_days = min(1.0, max(0.0, SAFETY_DAYS - days_cov) / SAFETY_DAYS)
        rows.append({
            "part":        p,
            "inv":         inv,
            "daily":       daily,
            "days_cov":    days_cov,
            "cat":         cat,
            "urgency_raw": gap_days,
        })

    if not rows:
        return {}, []

    max_daily = max(r["daily"] for r in rows) or 1.0
    scores, score_rows = {}, []

    for r in rows:
        p              = r["part"]
        urgency_score  = r["urgency_raw"] * 100
        category_score = CATEGORY_SCORE.get(r["cat"], 20)
        indent_score   = (r["daily"] / max_daily) * 100
        final_score    = (
            W_URGENCY  * urgency_score +
            W_CATEGORY * category_score +
            W_INDENT   * indent_score
        )
        scores[p] = round(final_score, 2)
        fixed_m    = part_fixed_machine.get(p, "—")
        score_rows.append({
            "Part":           p,
            "Category":       r["cat"],
            "Color":          part_color.get(p, "UNKNOWN"),
            "Fixed_Machine":  fixed_m,          # ← FIXED
            "Tools":          tools_available.get(p, 1),
            "Inventory_Now":  round(r["inv"], 0),
            "Daily_Indent":   round(r["daily"], 2),
            "Days_Coverage":  round(r["days_cov"], 2),
            "Safety_Floor":   SAFETY_DAYS,
            "Target_Ceiling": TARGET_DAYS,
            "Buffer_Status":  (
                "CRITICAL"     if r["days_cov"] < 1 else
                "BELOW_SAFETY" if r["days_cov"] < SAFETY_DAYS else
                "BUILDING"     if r["days_cov"] < TARGET_DAYS else
                "AT_TARGET"
            ),
            "Urgency_Score":  round(urgency_score, 1),
            "Category_Score": category_score,
            "Indent_Score":   round(indent_score, 1),
            "Final_Score":    round(final_score, 2),
        })

    return scores, score_rows

# =============================================================
# SECTION 13 — COLOUR-AWARE CHANGEOVER HELPER
# =============================================================

def _co_hrs_for(part, machine, machine_last_part):
    last = machine_last_part.get(machine)
    if last is None or last == part:
        return 0.0
    base_co    = vt_changeover.get(machine, DEFAULT_CHANGEOVER_HRS)
    last_color = part_color.get(last,  "UNKNOWN")
    new_color  = part_color.get(part,  "UNKNOWN")
    purge = (
        COLOR_PURGE_HRS
        if last_color != new_color
        and last_color != "UNKNOWN"
        and new_color  != "UNKNOWN"
        else 0.0
    )
    return base_co + purge

# =============================================================
# SECTION 14 — MACHINE RANKER  (fixed-machine aware)
# =============================================================
#
# Changes vs previous version:
#   • runner_lock is SUPPRESSED for fixed parts.
#     Fixed machine preference overrides Runner Lock.
#   • If the part has a fixed machine AND that machine is in
#     machines_to_try AND has effective capacity ≥ MIN_RUN_HOURS,
#     it is returned as rank[0] with a large cost penalty applied
#     to all other machines (ensuring fixed machine wins).
#   • If the fixed machine is not in machines_to_try or has
#     insufficient capacity, ranking proceeds normally over the
#     remaining machines.
# =============================================================

def rank_machines(part, machines_to_try, machine_hours,
                  machine_last_part, inv_days):
    category  = part_category.get(part, "Stranger")
    new_color = part_color.get(part, "UNKNOWN")

    # ── Fixed machine check ───────────────────────────────────
    # Runner Lock is suppressed for fixed parts.
    # For non-fixed parts, Runner Lock still applies.
    fixed_m = part_fixed_machine.get(part)
    is_fixed = fixed_m is not None

    if is_fixed:
        runner_lock = False   # fixed preference overrides Runner Lock
    else:
        runner_lock = (category == "Runner" and inv_days <= 1.0)

    # ── If part has a fixed machine, try it first ─────────────
    if is_fixed and fixed_m in machines_to_try:
        used_f = machine_hours.get(fixed_m, 0)
        free_f = round(AVAILABLE_HOURS - used_f, 4)
        co_f   = _co_hrs_for(part, fixed_m, machine_last_part)
        eff_f  = round(free_f - co_f, 4)

        if eff_f >= MIN_RUN_HOURS:
            # Fixed machine has capacity — return it first.
            # Still rank remaining machines normally as fallback
            # (they will only be used if caller explicitly ignores [0]).
            fallback = _rank_normal(
                part, [m for m in machines_to_try if m != fixed_m],
                machine_hours, machine_last_part, new_color, runner_lock
            )
            # Prepend fixed machine with cost = -1.0 (guaranteed lowest)
            return [(fixed_m, co_f, eff_f, -1.0)] + fallback, runner_lock

        # Fixed machine has no capacity — fall through to normal ranking
        # (fixed machine excluded since it has no room)
        remaining = [m for m in machines_to_try if m != fixed_m]
        return _rank_normal(
            part, remaining, machine_hours, machine_last_part,
            new_color, runner_lock
        ), runner_lock

    # ── Normal ranking (no fixed machine or fixed not applicable) ─
    return _rank_normal(
        part, machines_to_try, machine_hours, machine_last_part,
        new_color, runner_lock
    ), runner_lock


def _rank_normal(part, machines_to_try, machine_hours,
                 machine_last_part, new_color, runner_lock):
    """
    Standard cost-based ranking over the given machine list.
    Returns sorted list of (machine, co_hrs, effective_free, cost).
    """
    ranked = []
    for m in machines_to_try:
        used = machine_hours.get(m, 0)
        free = round(AVAILABLE_HOURS - used, 4)
        if free < MIN_RUN_HOURS:
            continue
        last = machine_last_part.get(m)
        if runner_lock and last != part:
            continue

        if last is None or last == part:
            co_hrs      = 0.0
            color_bonus = 0.0
        else:
            base_co    = vt_changeover.get(m, DEFAULT_CHANGEOVER_HRS)
            last_color = part_color.get(last, "UNKNOWN")
            same_color = (
                last_color == new_color
                and last_color != "UNKNOWN"
                and new_color  != "UNKNOWN"
            )
            purge      = (
                0.0 if same_color else (
                    COLOR_PURGE_HRS
                    if last_color != "UNKNOWN" and new_color != "UNKNOWN"
                    else 0.0
                )
            )
            co_hrs      = base_co + purge
            color_bonus = -0.08 if same_color else 0.0

        effective_free = round(free - co_hrs, 4)
        if effective_free < MIN_RUN_HOURS:
            continue

        part_count      = machine_part_count.get(m, max_part_count)
        count_score     = part_count / max_part_count
        co_penalty      = (co_hrs / AVAILABLE_HOURS) * 0.3
        util_penalty    = (used  / AVAILABLE_HOURS) * 0.2
        same_part_bonus = -0.15 if (last == part) else 0.0
        cost            = (count_score + co_penalty + util_penalty
                           + same_part_bonus + color_bonus)

        ranked.append((m, co_hrs, effective_free, cost))

    ranked.sort(key=lambda x: x[3])
    return ranked

# =============================================================
# SECTION 15 — TOOL-AWARE ASSIGNMENT
# =============================================================

def assign_part(part, scenario_id, machine_hours, machine_last_part,
                current_inventory, plan, already_planned,
                priority_scores):
    daily      = indent_daily.get(part, 0)
    monthly    = indent_monthly.get(part, 0)
    r_val      = rate.get(part, 1)
    inv_now    = current_inventory.get(part, 0)
    category   = part_category.get(part, "Stranger")
    tools      = tools_available.get(part, 1)
    score      = priority_scores.get(part, 0)
    color      = part_color.get(part, "UNKNOWN")
    inv_days   = inv_now / daily if daily > 0 else 999
    compatible = vt_compat.get(part, [])
    fixed_m    = part_fixed_machine.get(part)   # None if not fixed

    if not compatible:
        return []

    total_shortfall = max(0.0, daily - inv_now)
    hrs_for_full    = total_shortfall / r_val if r_val > 0 else MIN_RUN_HOURS
    hrs_for_full    = max(MIN_RUN_HOURS, hrs_for_full)

    new_rows        = []
    produced_so_far = 0.0
    tools_used      = 0
    used_machines   = set()

    ranked, runner_lock = rank_machines(
        part, compatible, machine_hours, machine_last_part, inv_days
    )
    if not ranked:
        return []

    m1, co1, eff1, _ = ranked[0]
    run1 = min(eff1, hrs_for_full)
    run1 = max(run1, MIN_RUN_HOURS)
    qty1 = round(run1 * r_val, 0)

    # Determine if fixed machine was used
    fixed_used = (fixed_m is not None and m1 == fixed_m)

    machine_hours[m1]       = round(machine_hours.get(m1, 0) + co1 + run1, 4)
    current_inventory[part] = round(current_inventory.get(part, 0) + qty1, 0)
    machine_last_part[m1]   = part
    produced_so_far        += qty1
    tools_used             += 1
    used_machines.add(m1)
    already_planned.add(part)

    indent_met_on_primary = (produced_so_far >= total_shortfall - 0.5)

    last_m1       = machine_state.get(m1)
    purge_applied = (
        last_m1 is not None and last_m1 != part
        and part_color.get(last_m1, "UNKNOWN") != color
        and part_color.get(last_m1, "UNKNOWN") != "UNKNOWN"
        and color != "UNKNOWN"
    )

    # Build type tag — include fixed machine flag
    type_tag = "Primary"
    if inv_now == 0:
        type_tag += " [ZERO-INV]"
    if fixed_used:
        type_tag += " [FIXED-MACHINE]"
    elif fixed_m is not None:
        type_tag += " [FIXED-FALLBACK]"   # fixed machine had no capacity

    new_rows.append({
        "Part":             part,
        "Color":            color,
        "Category":         category,
        "Fixed_Machine":    fixed_m or "—",          # ← FIXED
        "Fixed_Used":       "YES" if fixed_used else ("FALLBACK" if fixed_m else "N/A"),
        "Machine":          m1,
        "Run_Hours":        round(run1, 3),
        "Changeover_Hrs":   round(co1, 3),
        "Total_Hrs_Used":   round(co1 + run1, 3),
        "Rate_Per_Hour":    round(r_val, 2),
        "Production_Qty":   qty1,
        "Monthly_Indent":   round(monthly, 0),
        "Daily_Indent":     round(daily, 2),
        "Today_Target":     round(today_target_qty.get(part, 0), 0),
        "Changeover":       "No" if co1 == 0 else "Yes",
        "Color_Purge":      "Yes" if purge_applied else "No",
        "Type":             type_tag,
        "Role":             "Primary",
        "Tools_Available":  tools,
        "Tools_Used":       1,
        "Runner_Lock":      "YES" if runner_lock else "No",
        "Priority_Score":   score,
        "Phase":            1,
        "Indent_Met":       "YES" if indent_met_on_primary else "NO — shortfall remains",
        "Stagger_Adjusted": "No",
    })

    if not indent_met_on_primary:
        is_critical   = (inv_now == 0)
        tool_hard_cap = tools if is_critical else min(2, tools)

        while (produced_so_far < (total_shortfall - 0.5)
               and tools_used < tool_hard_cap):
            shortfall_now = total_shortfall - produced_so_far
            hrs_needed    = max(
                MIN_RUN_HOURS,
                shortfall_now / r_val if r_val > 0 else MIN_RUN_HOURS,
            )

            remaining_machines = [m for m in compatible if m not in used_machines]
            ranked_next, _ = rank_machines(
                part, remaining_machines, machine_hours,
                machine_last_part, inv_days
            )
            if not ranked_next:
                break

            mx, cox, effx, _ = ranked_next[0]
            run_x = min(effx, hrs_needed)
            run_x = max(run_x, MIN_RUN_HOURS)
            qty_x = round(run_x * r_val, 0)

            machine_hours[mx]       = round(machine_hours.get(mx, 0) + cox + run_x, 4)
            current_inventory[part] = round(current_inventory.get(part, 0) + qty_x, 0)
            machine_last_part[mx]   = part
            produced_so_far        += qty_x
            tools_used             += 1
            used_machines.add(mx)

            indent_met_here = (produced_so_far >= total_shortfall - 0.5)

            last_mx = machine_state.get(mx)
            purge_x = (
                last_mx is not None and last_mx != part
                and part_color.get(last_mx, "UNKNOWN") != color
                and part_color.get(last_mx, "UNKNOWN") != "UNKNOWN"
                and color != "UNKNOWN"
            )

            new_rows.append({
                "Part":             part,
                "Color":            color,
                "Category":         category,
                "Fixed_Machine":    fixed_m or "—",
                "Fixed_Used":       "N/A — expansion",
                "Machine":          mx,
                "Run_Hours":        round(run_x, 3),
                "Changeover_Hrs":   round(cox, 3),
                "Total_Hrs_Used":   round(cox + run_x, 3),
                "Rate_Per_Hour":    round(r_val, 2),
                "Production_Qty":   qty_x,
                "Monthly_Indent":   round(monthly, 0),
                "Daily_Indent":     round(daily, 2),
                "Today_Target":     round(today_target_qty.get(part, 0), 0),
                "Changeover":       "No" if cox == 0 else "Yes",
                "Color_Purge":      "Yes" if purge_x else "No",
                "Type":             "Tool-Expansion",
                "Role":             f"Tool-Expansion (tool {tools_used}/{tools})",
                "Tools_Available":  tools,
                "Tools_Used":       tools_used,
                "Runner_Lock":      "No",
                "Priority_Score":   score,
                "Phase":            2,
                "Indent_Met":       "YES" if indent_met_here else "NO — shortfall remains",
                "Stagger_Adjusted": "No",
            })
            crisis_tag = " [CRISIS — 3rd+ tool]" if tools_used >= 3 else ""
            print(f"      ↳ TOOL-EXP {part:26s} tool {tools_used}/{tool_hard_cap} → "
                  f"{mx:15s}  {run_x:.2f}h  qty={qty_x:.0f}  "
                  f"color={color}  "
                  f"{'COVERED ✓' if indent_met_here else 'still short'}"
                  f"{crisis_tag}")

    # Phase 3 — inventory build on same machines
    cap_days     = opd_cap(scenario_id)
    inv_after    = current_inventory.get(part, 0)
    cap_qty      = cap_days * daily
    headroom_qty = max(0.0, cap_qty - inv_after)

    if headroom_qty > 0:
        for row in new_rows:
            if headroom_qty <= 0:
                break
            m      = row["Machine"]
            free_m = round(AVAILABLE_HOURS - machine_hours.get(m, 0), 4)
            if free_m < 0.05:
                continue
            extend_hrs = min(
                free_m,
                headroom_qty / r_val if r_val > 0 else 0,
            )
            if extend_hrs < 0.05:
                continue
            extra_qty = round(extend_hrs * r_val, 0)

            row["Run_Hours"]      = round(float(row["Run_Hours"]) + extend_hrs, 3)
            row["Total_Hrs_Used"] = round(
                float(row["Changeover_Hrs"]) + float(row["Run_Hours"]), 3
            )
            row["Production_Qty"] = round(float(row["Production_Qty"]) + extra_qty, 0)
            row["Type"]           = str(row["Type"]) + "+InvBuild"

            machine_hours[m]        = round(machine_hours.get(m, 0) + extend_hrs, 4)
            current_inventory[part] = round(current_inventory.get(part, 0) + extra_qty, 0)
            headroom_qty           -= extra_qty

            print(f"      ↳ INV-BUILD {part:25s} on {m:15s}  "
                  f"+{extend_hrs:.2f}h  qty+={extra_qty:.0f}  "
                  f"(target ceiling: {cap_days:.1f} days)")

    for row in new_rows:
        row["Tools_Used"] = tools_used

    return new_rows

# =============================================================
# SECTION 16 — RUNNER PRIORITY ENFORCEMENT
# =============================================================

def enforce_runner_priority(plan, machine_hours, machine_last_part,
                             current_inventory, already_planned,
                             priority_scores):
    print(f"\n{'─'*65}")
    print(f"  RUNNER PRIORITY ENFORCEMENT")
    print(f"  Rule: Runner with inv < daily_indent displaces Strangers/Repeaters")
    print(f"  Victim sort: lowest daily_indent first, highest planned_qty first")
    print(f"{'─'*65}")

    runner_priority_log = []

    critical_runners = []
    for part in vt_compat:
        if part in already_planned:
            continue
        if part_category.get(part, "Stranger") != "Runner":
            continue
        daily = indent_daily.get(part, 0)
        inv   = current_inventory.get(part, 0)
        r_val = rate.get(part, 1)
        if daily <= 0 or r_val <= 0:
            continue
        if inv >= daily:
            continue
        skip, _ = should_skip(part)
        if skip:
            continue
        if not vt_compat.get(part):
            continue
        critical_runners.append(part)

    if not critical_runners:
        print(f"  No critical unplanned Runners found  ✓")
        return runner_priority_log

    critical_runners.sort(key=lambda p: indent_daily.get(p, 0), reverse=True)
    print(f"  Critical Runners to enforce: {len(critical_runners)}")
    for p in critical_runners:
        inv   = current_inventory.get(p, 0)
        daily = indent_daily.get(p, 0)
        fm    = part_fixed_machine.get(p, "—")
        print(f"    {p:<30}  inv={inv:.0f}  daily={daily:.2f}  "
              f"gap={daily - inv:.2f} pcs  fixed={fm}")

    for runner in critical_runners:
        r_daily   = indent_daily.get(runner, 0)
        r_inv     = current_inventory.get(runner, 0)
        r_rate    = rate.get(runner, 1)
        r_color   = part_color.get(runner, "UNKNOWN")
        r_score   = priority_scores.get(runner, 0)
        r_monthly = indent_monthly.get(runner, 0)
        fixed_m   = part_fixed_machine.get(runner)

        shortfall_qty = max(0.0, r_daily - r_inv)
        hours_needed  = max(
            MIN_RUN_HOURS,
            shortfall_qty / r_rate if r_rate > 0 else MIN_RUN_HOURS,
        )

        # Build candidate machine list:
        # If runner has a fixed machine, try it first, then the rest.
        compatible_machines = vt_compat.get(runner, [])
        if fixed_m and fixed_m in compatible_machines:
            ordered_machines = [fixed_m] + [
                m for m in compatible_machines if m != fixed_m
            ]
        else:
            ordered_machines = compatible_machines

        print(f"\n  Runner: {runner}  needs {hours_needed:.2f}h  "
              f"(shortfall={shortfall_qty:.0f} pcs  "
              f"rate={r_rate:.1f}/h  inv={r_inv:.0f}  daily={r_daily:.2f}  "
              f"fixed={fixed_m or '—'})")

        best_machine    = None
        best_co_hrs     = 0.0
        best_victims    = []
        best_disruption = float("inf")

        for m in ordered_machines:
            co_hrs   = _co_hrs_for(runner, m, machine_last_part)
            used_hrs = machine_hours.get(m, 0)
            free_hrs = round(AVAILABLE_HOURS - used_hrs, 4)
            eff_free = round(free_hrs - co_hrs, 4)

            # Free capacity — no displacement needed
            if eff_free >= hours_needed:
                run_h = min(eff_free, hours_needed)
                run_h = max(run_h, MIN_RUN_HOURS)
                qty   = round(run_h * r_rate, 0)

                last_m    = machine_last_part.get(m)
                purge     = (
                    last_m is not None and last_m != runner
                    and part_color.get(last_m, "UNKNOWN") != r_color
                    and part_color.get(last_m, "UNKNOWN") != "UNKNOWN"
                    and r_color != "UNKNOWN"
                )
                fixed_used = (fixed_m is not None and m == fixed_m)

                machine_hours[m]          = round(used_hrs + co_hrs + run_h, 4)
                current_inventory[runner] = round(
                    current_inventory.get(runner, 0) + qty, 0
                )
                machine_last_part[m]      = runner
                already_planned.add(runner)

                plan.append({
                    "Part":             runner,
                    "Color":            r_color,
                    "Category":         "Runner",
                    "Fixed_Machine":    fixed_m or "—",
                    "Fixed_Used":       "YES" if fixed_used else ("FALLBACK" if fixed_m else "N/A"),
                    "Machine":          m,
                    "Run_Hours":        round(run_h, 3),
                    "Changeover_Hrs":   round(co_hrs, 3),
                    "Total_Hrs_Used":   round(co_hrs + run_h, 3),
                    "Rate_Per_Hour":    round(r_rate, 2),
                    "Production_Qty":   qty,
                    "Monthly_Indent":   round(r_monthly, 0),
                    "Daily_Indent":     round(r_daily, 2),
                    "Today_Target":     round(today_target_qty.get(runner, 0), 0),
                    "Changeover":       "No" if co_hrs == 0 else "Yes",
                    "Color_Purge":      "Yes" if purge else "No",
                    "Type":             "Runner-Priority [SUB-1-DAY] (free capacity)"
                                        + (" [FIXED-MACHINE]" if fixed_used else ""),
                    "Role":             "Primary",
                    "Tools_Available":  tools_available.get(runner, 1),
                    "Tools_Used":       1,
                    "Runner_Lock":      "No",
                    "Priority_Score":   r_score,
                    "Phase":            1,
                    "Indent_Met":       "YES" if qty >= shortfall_qty else "NO — partial",
                    "Stagger_Adjusted": "No",
                })

                runner_priority_log.append({
                    "Runner_Part":          runner,
                    "Fixed_Machine":        fixed_m or "—",
                    "Fixed_Used":           "YES" if fixed_used else ("FALLBACK" if fixed_m else "N/A"),
                    "Runner_Daily_Indent":  round(r_daily, 2),
                    "Runner_Inv_Before":    round(r_inv, 0),
                    "Runner_Shortfall":     round(shortfall_qty, 0),
                    "Machine_Assigned":     m,
                    "Hours_Needed":         round(hours_needed, 3),
                    "Hours_Assigned":       round(run_h, 3),
                    "Qty_Produced":         qty,
                    "Displacement_Used":    "No — free capacity available",
                    "Victims":              "—",
                    "Total_Hours_Reclaimed":"—",
                    "Result":               "PLANNED — free capacity",
                })

                print(f"    → {runner} placed on {m} (free capacity)  "
                      f"run={run_h:.2f}h  qty={qty:.0f}  "
                      f"fixed={'YES' if fixed_used else 'FALLBACK'}  ✓")
                best_machine = "DONE"
                break

            # Full machine — try displacement
            machine_rows = [r for r in plan if r["Machine"] == m]
            yieldable    = [
                r for r in machine_rows
                if part_category.get(r["Part"], "Stranger") in ("Stranger", "Repeater")
            ]
            if not yieldable:
                continue

            yieldable.sort(key=lambda r: (
                indent_daily.get(r["Part"], 0),
                -float(r.get("Production_Qty", 0)),
            ))

            reclaimable_detail = []
            for row in yieldable:
                row_run = float(row.get("Run_Hours", 0))
                if row_run <= 0:
                    continue
                if row_run > MIN_RUN_HOURS:
                    reclaimable_detail.append(
                        (row, round(row_run - MIN_RUN_HOURS, 4), "partial")
                    )
                else:
                    reclaimable_detail.append(
                        (row, round(row_run, 4), "full_remove")
                    )

            total_reclaimable        = sum(x[1] for x in reclaimable_detail)
            runner_eff_after_reclaim = round(
                free_hrs + total_reclaimable - co_hrs, 4
            )

            if runner_eff_after_reclaim < MIN_RUN_HOURS:
                continue
            if runner_eff_after_reclaim < hours_needed:
                continue

            disruption = total_reclaimable
            if disruption < best_disruption:
                best_disruption = disruption
                best_machine    = m
                best_co_hrs     = co_hrs
                best_victims    = reclaimable_detail

        # Execute displacement
        if best_machine is None:
            print(f"    ✗ {runner}  — no machine can provide {hours_needed:.2f}h  "
                  f"→ logged as unresolvable")
            runner_priority_log.append({
                "Runner_Part":          runner,
                "Fixed_Machine":        fixed_m or "—",
                "Fixed_Used":           "FAILED",
                "Runner_Daily_Indent":  round(r_daily, 2),
                "Runner_Inv_Before":    round(r_inv, 0),
                "Runner_Shortfall":     round(shortfall_qty, 0),
                "Machine_Assigned":     "—",
                "Hours_Needed":         round(hours_needed, 3),
                "Hours_Assigned":       0,
                "Qty_Produced":         0,
                "Displacement_Used":    "N/A",
                "Victims":              "—",
                "Total_Hours_Reclaimed":"—",
                "Result":               "FAILED — insufficient capacity on all compatible machines",
            })
            continue

        if best_machine == "DONE":
            continue

        # Carve hours from victims
        hours_to_free    = hours_needed
        victim_log_parts = []
        total_reclaimed  = 0.0

        for (victim_row, reclaimable_hrs, reclaim_type) in best_victims:
            if hours_to_free <= 0.001:
                break

            vpart      = victim_row["Part"]
            v_rate     = rate.get(vpart, 1)
            v_run_orig = float(victim_row.get("Run_Hours", 0))
            carve_hrs  = round(min(reclaimable_hrs, hours_to_free), 4)
            if carve_hrs <= 0:
                continue

            new_run_hrs = round(v_run_orig - carve_hrs, 4)

            if new_run_hrs < MIN_RUN_HOURS:
                plan.remove(victim_row)
                machine_hours[best_machine] = round(
                    machine_hours.get(best_machine, 0) - v_run_orig, 4
                )
                lost_qty = round(v_run_orig * v_rate, 0)
                current_inventory[vpart] = round(
                    current_inventory.get(vpart, 0) - lost_qty, 0
                )
                actually_freed = v_run_orig
                victim_log_parts.append(
                    f"{vpart} REMOVED ({v_run_orig:.2f}h / {lost_qty:.0f} pcs lost)"
                )
                print(f"    ↳ YIELD (remove)  {vpart:28s}  "
                      f"freed {v_run_orig:.2f}h  lost {lost_qty:.0f} pcs")
            else:
                lost_qty  = round(carve_hrs * v_rate, 0)
                new_qty   = round(new_run_hrs * v_rate, 0)
                new_total = round(
                    float(victim_row.get("Changeover_Hrs", 0)) + new_run_hrs, 3
                )
                victim_row["Run_Hours"]      = new_run_hrs
                victim_row["Production_Qty"] = new_qty
                victim_row["Total_Hrs_Used"] = new_total
                victim_row["Type"]           = (
                    str(victim_row.get("Type", "Primary")) + " [YIELDED_TO_RUNNER]"
                )
                machine_hours[best_machine] = round(
                    machine_hours.get(best_machine, 0) - carve_hrs, 4
                )
                current_inventory[vpart] = round(
                    current_inventory.get(vpart, 0) - lost_qty, 0
                )
                actually_freed = carve_hrs
                victim_log_parts.append(
                    f"{vpart} reduced by {carve_hrs:.2f}h "
                    f"(−{lost_qty:.0f} pcs  remaining={new_qty:.0f})"
                )
                print(f"    ↳ YIELD (reduce)  {vpart:28s}  "
                      f"−{carve_hrs:.2f}h  −{lost_qty:.0f} pcs  "
                      f"remaining={new_qty:.0f} pcs")

            hours_to_free   = round(hours_to_free - actually_freed, 4)
            total_reclaimed = round(total_reclaimed + actually_freed, 4)

        # Assign runner to freed machine
        used_now = machine_hours.get(best_machine, 0)
        free_now = round(AVAILABLE_HOURS - used_now, 4)
        eff_free = round(free_now - best_co_hrs, 4)

        if eff_free < MIN_RUN_HOURS:
            print(f"    ✗ {runner}  safety check failed after carve  → skip")
            runner_priority_log.append({
                "Runner_Part":          runner,
                "Fixed_Machine":        fixed_m or "—",
                "Fixed_Used":           "FAILED",
                "Runner_Daily_Indent":  round(r_daily, 2),
                "Runner_Inv_Before":    round(r_inv, 0),
                "Runner_Shortfall":     round(shortfall_qty, 0),
                "Machine_Assigned":     best_machine,
                "Hours_Needed":         round(hours_needed, 3),
                "Hours_Assigned":       0,
                "Qty_Produced":         0,
                "Displacement_Used":    "Yes",
                "Victims":              "; ".join(victim_log_parts),
                "Total_Hours_Reclaimed":round(total_reclaimed, 3),
                "Result":               "FAILED — safety check after carve",
            })
            continue

        run_hrs    = max(MIN_RUN_HOURS, min(eff_free, hours_needed))
        qty        = round(run_hrs * r_rate, 0)
        last_m     = machine_last_part.get(best_machine)
        purge      = (
            last_m is not None and last_m != runner
            and part_color.get(last_m, "UNKNOWN") != r_color
            and part_color.get(last_m, "UNKNOWN") != "UNKNOWN"
            and r_color != "UNKNOWN"
        )
        fixed_used = (fixed_m is not None and best_machine == fixed_m)

        machine_hours[best_machine]     = round(used_now + best_co_hrs + run_hrs, 4)
        current_inventory[runner]       = round(
            current_inventory.get(runner, 0) + qty, 0
        )
        machine_last_part[best_machine] = runner
        already_planned.add(runner)

        plan.append({
            "Part":             runner,
            "Color":            r_color,
            "Category":         "Runner",
            "Fixed_Machine":    fixed_m or "—",
            "Fixed_Used":       "YES" if fixed_used else ("FALLBACK" if fixed_m else "N/A"),
            "Machine":          best_machine,
            "Run_Hours":        round(run_hrs, 3),
            "Changeover_Hrs":   round(best_co_hrs, 3),
            "Total_Hrs_Used":   round(best_co_hrs + run_hrs, 3),
            "Rate_Per_Hour":    round(r_rate, 2),
            "Production_Qty":   qty,
            "Monthly_Indent":   round(r_monthly, 0),
            "Daily_Indent":     round(r_daily, 2),
            "Today_Target":     round(today_target_qty.get(runner, 0), 0),
            "Changeover":       "No" if best_co_hrs == 0 else "Yes",
            "Color_Purge":      "Yes" if purge else "No",
            "Type":             "Runner-Priority [SUB-1-DAY] [DISPLACED]"
                                + (" [FIXED-MACHINE]" if fixed_used else ""),
            "Role":             "Primary",
            "Tools_Available":  tools_available.get(runner, 1),
            "Tools_Used":       1,
            "Runner_Lock":      "No",
            "Priority_Score":   r_score,
            "Phase":            1,
            "Indent_Met":       "YES" if qty >= shortfall_qty else "NO — partial",
            "Stagger_Adjusted": "No",
        })

        runner_priority_log.append({
            "Runner_Part":          runner,
            "Fixed_Machine":        fixed_m or "—",
            "Fixed_Used":           "YES" if fixed_used else ("FALLBACK" if fixed_m else "N/A"),
            "Runner_Daily_Indent":  round(r_daily, 2),
            "Runner_Inv_Before":    round(r_inv, 0),
            "Runner_Shortfall":     round(shortfall_qty, 0),
            "Machine_Assigned":     best_machine,
            "Hours_Needed":         round(hours_needed, 3),
            "Hours_Assigned":       round(run_hrs, 3),
            "Qty_Produced":         qty,
            "Displacement_Used":    "Yes",
            "Victims":              "; ".join(victim_log_parts),
            "Total_Hours_Reclaimed":round(total_reclaimed, 3),
            "Result":               (
                "PLANNED — displacement successful"
                if qty >= shortfall_qty - 0.5
                else "PLANNED — partial (machine hours limited)"
            ),
        })

        print(f"    ✓ {runner:30s} → {best_machine}  "
              f"run={run_hrs:.2f}h  qty={qty:.0f}  "
              f"reclaimed={total_reclaimed:.2f}h  "
              f"fixed={'YES' if fixed_used else 'FALLBACK'}  "
              f"purge={'Yes' if purge else 'No'}")

    total_placed = sum(
        1 for r in runner_priority_log if "PLANNED" in r.get("Result", "")
    )
    total_failed = sum(
        1 for r in runner_priority_log if "FAILED" in r.get("Result", "")
    )
    print(f"\n  Runner Priority Enforcement complete: "
          f"{total_placed} placed  |  {total_failed} unresolvable")

    return runner_priority_log

# =============================================================
# SECTION 17 — DISPLACEMENT PRE-PASS  (V9 original)
# =============================================================

def displace_for_zero_inv(part, machine_hours, machine_last_part,
                           current_inventory, plan, already_planned,
                           priority_scores):
    daily      = indent_daily.get(part, 0)
    r_val      = rate.get(part, 1)
    category   = part_category.get(part, "Stranger")
    score      = priority_scores.get(part, 0)
    compatible = vt_compat.get(part, [])
    fixed_m    = part_fixed_machine.get(part)

    if not compatible:
        return False

    candidate_machines = [
        m for m in compatible
        if round(AVAILABLE_HOURS - machine_hours.get(m, 0), 4) < MIN_RUN_HOURS
    ]
    if not candidate_machines:
        return False

    best_machine     = None
    best_victim_row  = None
    best_victim_days = -1

    for m in candidate_machines:
        for row in [r for r in plan if r["Machine"] == m]:
            vpart  = row["Part"]
            vdaily = indent_daily.get(vpart, 0)
            vinv   = current_inventory.get(vpart, 0)
            vdays  = vinv / vdaily if vdaily > 0 else 999
            if vdays < SAFETY_DAYS or vinv <= 0:
                continue
            vrun = float(row.get("Run_Hours", 0))
            if vrun - MIN_RUN_HOURS < MIN_RUN_HOURS:
                continue
            if vdays > best_victim_days:
                best_victim_days = vdays
                best_victim_row  = row
                best_machine     = m

    if best_machine is None or best_victim_row is None:
        return False

    vpart    = best_victim_row["Part"]
    vr_val   = rate.get(vpart, 1)
    reduce_h = MIN_RUN_HOURS
    lost_qty = round(reduce_h * vr_val, 0)

    best_victim_row["Run_Hours"]      = round(
        float(best_victim_row["Run_Hours"]) - reduce_h, 3
    )
    best_victim_row["Production_Qty"] = round(
        float(best_victim_row["Production_Qty"]) - lost_qty, 0
    )
    best_victim_row["Total_Hrs_Used"] = round(
        float(best_victim_row.get("Changeover_Hrs", 0))
        + float(best_victim_row["Run_Hours"]), 3
    )
    best_victim_row["Type"] = (
        str(best_victim_row.get("Type", "Primary")) + " [DISPLACED]"
    )

    current_inventory[vpart]    = round(
        current_inventory.get(vpart, 0) - lost_qty, 0
    )
    machine_hours[best_machine] = round(
        machine_hours.get(best_machine, 0) - reduce_h, 4
    )

    last  = machine_last_part.get(best_machine)
    p_col = part_color.get(part, "UNKNOWN")
    l_col = part_color.get(last, "UNKNOWN") if last else "UNKNOWN"
    if last is None or last == part:
        co_hrs = 0.0
    else:
        base_co = vt_changeover.get(best_machine, DEFAULT_CHANGEOVER_HRS)
        purge   = (
            COLOR_PURGE_HRS
            if p_col != l_col and p_col != "UNKNOWN" and l_col != "UNKNOWN"
            else 0.0
        )
        co_hrs = base_co + purge

    eff_free = round(
        AVAILABLE_HOURS - machine_hours.get(best_machine, 0) - co_hrs, 4
    )
    run_hrs  = max(MIN_RUN_HOURS, min(eff_free, MIN_RUN_HOURS))
    qty      = round(run_hrs * r_val, 0)

    machine_hours[best_machine]     = round(
        machine_hours.get(best_machine, 0) + co_hrs + run_hrs, 4
    )
    current_inventory[part]         = round(
        current_inventory.get(part, 0) + qty, 0
    )
    machine_last_part[best_machine] = part
    already_planned.add(part)

    has_purge  = (
        last is not None and last != part
        and p_col != l_col
        and p_col != "UNKNOWN" and l_col != "UNKNOWN"
    )
    fixed_used = (fixed_m is not None and best_machine == fixed_m)

    plan.append({
        "Part":             part,
        "Color":            p_col,
        "Category":         category,
        "Fixed_Machine":    fixed_m or "—",
        "Fixed_Used":       "YES" if fixed_used else ("FALLBACK" if fixed_m else "N/A"),
        "Machine":          best_machine,
        "Run_Hours":        round(run_hrs, 3),
        "Changeover_Hrs":   round(co_hrs, 3),
        "Total_Hrs_Used":   round(co_hrs + run_hrs, 3),
        "Rate_Per_Hour":    round(r_val, 2),
        "Production_Qty":   qty,
        "Monthly_Indent":   round(indent_monthly.get(part, 0), 0),
        "Daily_Indent":     round(daily, 2),
        "Today_Target":     round(today_target_qty.get(part, 0), 0),
        "Changeover":       "No" if co_hrs == 0 else "Yes",
        "Color_Purge":      "Yes" if has_purge else "No",
        "Type":             "Displacement [ZERO-INV PRIORITY]",
        "Role":             "Primary",
        "Tools_Available":  tools_available.get(part, 1),
        "Tools_Used":       1,
        "Runner_Lock":      "No",
        "Priority_Score":   score,
        "Phase":            1,
        "Indent_Met":       "YES" if qty >= daily else "NO — partial",
        "Stagger_Adjusted": "No",
        "Displaced_Victim": vpart,
        "Victim_Days_Stock":round(best_victim_days, 2),
    })

    print(f"      ↳ DISPLACEMENT  {part:26s} → {best_machine:15s}  "
          f"freed from {vpart} ({best_victim_days:.1f} days stock)  "
          f"run={run_hrs:.2f}h  qty={qty:.0f}  "
          f"fixed={'YES' if fixed_used else 'N/A'}")
    return True

# =============================================================
# SECTION 18 — TOOL-CHANGER HELPERS
# =============================================================

def _fmt_h(h):
    try:
        total_min = int(round(float(h) * 60))
        return f"{total_min // 60:02d}:{total_min % 60:02d}"
    except Exception:
        return "??"


def _collect_co_events(plan, machines):
    events = []
    for m in machines:
        m_rows = [r for r in plan if r["Machine"] == m]
        if len(m_rows) < 2:
            continue
        cursor = 0.0
        for i, row in enumerate(m_rows):
            co_h  = float(row.get("Changeover_Hrs") or 0.0)
            run_h = float(row.get("Run_Hours")       or 0.0)
            if co_h > 0 and i > 0:
                events.append({
                    "machine":       m,
                    "part_before":   m_rows[i - 1]["Part"],
                    "part_after":    row["Part"],
                    "co_duration":   co_h,
                    "natural_start": cursor,
                    "row_before":    m_rows[i - 1],
                    "row_after":     row,
                    "actual_start":  None,
                    "wait_hrs":      0.0,
                })
            cursor += co_h + run_h
    return events


def _recompute_natural_start(ev, plan):
    m, target = ev["machine"], ev["row_after"]
    cursor = 0.0
    for r in [r for r in plan if r["Machine"] == m]:
        if r is target:
            break
        cursor += (float(r.get("Changeover_Hrs") or 0)
                   + float(r.get("Run_Hours") or 0))
    return cursor


def _machine_spare(m, plan):
    used = sum(
        float(r.get("Changeover_Hrs") or 0) + float(r.get("Run_Hours") or 0)
        for r in plan if r["Machine"] == m
    )
    return max(0.0, AVAILABLE_HOURS - used)


def _extend_row_before(ev, wait_hrs, plan):
    spare     = _machine_spare(ev["machine"], plan)
    extend_by = min(wait_hrs, spare)
    if extend_by <= 0:
        return 0.0, 0
    rb    = ev["row_before"]
    r_val = rate.get(rb["Part"], 1.0)
    extra = round(extend_by * r_val, 0)
    rb["Run_Hours"]      = round(float(rb.get("Run_Hours") or 0) + extend_by, 3)
    rb["Production_Qty"] = round(float(rb.get("Production_Qty") or 0) + extra, 0)
    rb["Total_Hrs_Used"] = round(
        float(rb.get("Changeover_Hrs") or 0) + float(rb["Run_Hours"]), 3
    )
    rb["Stagger_Adjusted"] = (
        f"CO: extended +{round(extend_by * 60, 1)}min to fill TC wait"
    )
    return extend_by, extra

# =============================================================
# SECTION 19 — QUANTITY-BASED CO STAGGER
# =============================================================

MIN_CO_GAP_HRS = 20 / 60.0


def _finish_time_of_co(ev, plan):
    m, target = ev["machine"], ev["row_after"]
    total = 0.0
    for row in plan:
        if row["Machine"] != m:
            continue
        if row is target:
            break
        total += float(row.get("Run_Hours") or 0)
        total += float(row.get("Changeover_Hrs") or 0)
    return round(total, 4)


def _last_row_before_co(ev, plan):
    return ev["row_before"]


def stagger_co_by_quantity(plan, machines, scenario_id, current_inventory):
    events = _collect_co_events(plan, machines)
    if len(events) < 2:
        return 0

    adjustments = 0
    max_passes  = len(events) * 2

    for _ in range(max_passes):
        for ev in events:
            ev["_ft"] = _finish_time_of_co(ev, plan)
        events.sort(key=lambda e: e["_ft"])

        conflict = None
        for i in range(len(events) - 1):
            co_dur_e = events[i]["co_duration"]
            required = co_dur_e + MIN_CO_GAP_HRS
            gap      = events[i + 1]["_ft"] - events[i]["_ft"]
            if gap < required - 0.001:
                conflict = (events[i], events[i + 1], gap, required)
                break

        if conflict is None:
            break

        ev_early, ev_late, gap, required_gap = conflict
        shortfall_hrs = required_gap - gap

        row_late   = _last_row_before_co(ev_late, plan)
        p_late     = row_late["Part"]
        r_late     = rate.get(p_late, 1)
        m_late     = ev_late["machine"]
        daily_l    = indent_daily.get(p_late, 0)
        inv_l      = current_inventory.get(p_late, 0)
        cap_qty    = opd_cap(scenario_id) * daily_l
        produced_l = float(row_late.get("Production_Qty") or 0)
        headroom   = max(0.0, cap_qty - inv_l)
        push_hrs   = shortfall_hrs
        push_qty   = round(push_hrs * r_late, 0)
        used_m     = sum(
            float(r.get("Changeover_Hrs") or 0) + float(r.get("Run_Hours") or 0)
            for r in plan if r["Machine"] == m_late
        )
        free_m     = max(0.0, AVAILABLE_HOURS - used_m)
        can_push   = (
            push_qty <= headroom
            and push_hrs <= free_m + 0.001
            and r_late > 0
        )

        if can_push:
            row_late["Run_Hours"]      = round(
                float(row_late.get("Run_Hours") or 0) + push_hrs, 3
            )
            row_late["Production_Qty"] = round(produced_l + push_qty, 0)
            row_late["Total_Hrs_Used"] = round(
                float(row_late.get("Changeover_Hrs") or 0)
                + float(row_late["Run_Hours"]), 3
            )
            row_late["Stagger_Adjusted"] = (
                f"CO-stagger PUSH +{round(push_hrs * 60, 1)}min "
                f"gap={round(gap * 60, 1)}min "
                f"need={round(required_gap * 60, 0):.0f}min"
            )
            current_inventory[p_late] = round(
                current_inventory.get(p_late, 0) + push_qty, 0
            )
            adjustments += 1
            continue

        row_early  = _last_row_before_co(ev_early, plan)
        p_early    = row_early["Part"]
        r_early    = rate.get(p_early, 1)
        daily_e    = indent_daily.get(p_early, 0)
        inv_e      = current_inventory.get(p_early, 0)
        produced_e = float(row_early.get("Production_Qty") or 0)
        min_qty_e  = max(0.0, daily_e - inv_e)
        max_pull   = max(0.0, produced_e - min_qty_e)
        pull_hrs   = shortfall_hrs
        pull_qty   = round(pull_hrs * r_early, 0)
        can_pull   = (
            pull_qty <= max_pull
            and r_early > 0
            and produced_e - pull_qty >= MIN_RUN_HOURS * r_early
        )

        if can_pull:
            row_early["Run_Hours"]      = round(
                float(row_early.get("Run_Hours") or 0) - pull_hrs, 3
            )
            row_early["Production_Qty"] = round(produced_e - pull_qty, 0)
            row_early["Total_Hrs_Used"] = round(
                float(row_early.get("Changeover_Hrs") or 0)
                + float(row_early["Run_Hours"]), 3
            )
            row_early["Stagger_Adjusted"] = (
                f"CO-stagger PULL -{round(pull_hrs * 60, 1)}min "
                f"gap={round(gap * 60, 1)}min "
                f"need={round(required_gap * 60, 0):.0f}min"
            )
            current_inventory[p_early] = round(
                current_inventory.get(p_early, 0) - pull_qty, 0
            )
            adjustments += 1
            continue

        print(f"    [CO-STAGGER SKIP]  {ev_early['machine']} / {ev_late['machine']}  "
              f"gap={round(gap * 60, 1)}min  "
              f"need={round(required_gap * 60, 0):.0f}min  unresolvable")
        ev_late["_ft"] = ev_early["_ft"] + MIN_CO_GAP_HRS
        break

    return adjustments


def stagger_changeovers_serial_queue(plan, machines):
    print(f"\n  Tool-Changer Serial Queue Scheduler")
    print(f"  Guarantee: strict serial — no two COs can overlap by construction")

    events = _collect_co_events(plan, machines)
    if not events:
        print(f"  No changeovers in plan — tool changer idle  ✓")
        return

    events.sort(key=lambda e: e["natural_start"])
    print(f"  {len(events)} CO events queued across "
          f"{len({e['machine'] for e in events})} machines")
    print()
    print(f"  {'#':<4} {'Machine':<18} {'Part Before':<22} {'Part After':<22} "
          f"{'Dur':>5} {'Natural':>8} {'Actual':>8} {'Wait':>7} {'Fill':>6} {'Purge':>6}")
    print(f"  {'─'*105}")

    tool_changer_free_at = 0.0
    total_extra_pcs      = 0
    total_wait_min       = 0.0

    for idx, ev in enumerate(events, 1):
        natural_start        = _recompute_natural_start(ev, plan)
        co_h                 = ev["co_duration"]
        actual_start         = max(natural_start, tool_changer_free_at)
        wait_hrs             = round(actual_start - natural_start, 4)
        tool_changer_free_at = actual_start + co_h

        extra_pcs = 0
        if wait_hrs > 0.001:
            _, extra_pcs     = _extend_row_before(ev, wait_hrs, plan)
            total_extra_pcs += extra_pcs
            total_wait_min  += wait_hrs * 60

        ev["actual_start"] = actual_start
        ev["wait_hrs"]     = wait_hrs

        wait_str     = f"+{round(wait_hrs * 60, 1)}m" if wait_hrs > 0.001 else "none"
        fill_str     = f"+{extra_pcs:.0f}" if extra_pcs > 0 else "—"
        before_color = part_color.get(ev["part_before"], "?")
        after_color  = part_color.get(ev["part_after"],  "?")
        purge_str    = (
            "PURGE"
            if before_color != after_color
            and before_color not in ("?", "UNKNOWN")
            and after_color  not in ("?", "UNKNOWN")
            else "—"
        )

        print(f"  {idx:<4} {ev['machine']:<18} {ev['part_before']:<22} "
              f"{ev['part_after']:<22} "
              f"{round(co_h * 60, 1):>4.0f}m "
              f"{_fmt_h(natural_start):>8} "
              f"{_fmt_h(actual_start):>8} "
              f"{wait_str:>7} "
              f"{fill_str:>6} "
              f"{purge_str:>6}")

    n_waited = sum(1 for e in events if e.get("wait_hrs", 0) > 0.001)
    print(f"\n  Queue complete. Tool changer free at: {_fmt_h(tool_changer_free_at)}")
    print(f"  Events that had to wait : {n_waited} / {len(events)}")
    print(f"  Total wait time filled  : {round(total_wait_min, 1)} min")
    print(f"  Extra pieces produced   : {total_extra_pcs:,.0f}")
    print(f"  Overlap guarantee       : ABSOLUTE — serial queue")


def stagger_changeovers(plan, machines):
    stagger_changeovers_serial_queue(plan, machines)

# =============================================================
# SECTION 20 — 22H UTILISATION ENFORCER
# =============================================================

def _add_part_to_machine(p, m, run_hrs, co_hrs, machine_hours,
                          machine_last_part, current_inventory,
                          already_planned, plan, priority_scores,
                          type_label, role_label):
    r_val   = rate.get(p, 1)
    qty     = round(run_hrs * r_val, 0)
    last    = machine_last_part.get(m)
    p_color = part_color.get(p,    "UNKNOWN")
    l_color = part_color.get(last, "UNKNOWN") if last else "UNKNOWN"
    has_purge = (
        last is not None and last != p
        and p_color != l_color
        and p_color != "UNKNOWN" and l_color != "UNKNOWN"
    )

    machine_hours[m]     = round(machine_hours.get(m, 0) + co_hrs + run_hrs, 4)
    current_inventory[p] = round(current_inventory.get(p, 0) + qty, 0)
    machine_last_part[m] = p
    already_planned.add(p)

    plan.append({
        "Part":             p,
        "Color":            p_color,
        "Category":         part_category.get(p, "Stranger"),
        "Fixed_Machine":    part_fixed_machine.get(p, "—"),
        "Fixed_Used":       "N/A — enforcer",
        "Machine":          m,
        "Run_Hours":        round(run_hrs, 3),
        "Changeover_Hrs":   round(co_hrs, 3),
        "Total_Hrs_Used":   round(co_hrs + run_hrs, 3),
        "Rate_Per_Hour":    round(r_val, 2),
        "Production_Qty":   qty,
        "Monthly_Indent":   round(indent_monthly.get(p, 0), 0),
        "Daily_Indent":     round(indent_daily.get(p, 0), 2),
        "Today_Target":     round(today_target_qty.get(p, 0), 0),
        "Changeover":       "No" if co_hrs == 0 else "Yes",
        "Color_Purge":      "Yes" if has_purge else "No",
        "Type":             type_label,
        "Role":             role_label,
        "Tools_Available":  tools_available.get(p, 1),
        "Tools_Used":       1,
        "Runner_Lock":      "No",
        "Priority_Score":   round(priority_scores.get(p, 0), 2),
        "Phase":            1,
        "Stagger_Adjusted": "No",
    })
    return qty


def utilization_enforcer(plan, machine_hours, machine_last_part,
                          all_parts, already_planned,
                          current_inventory, scenario_id,
                          priority_scores):
    print(f"\n  22H UTILIZATION ENFORCER  (minimum floor = {UTIL_TARGET_PCT}%)")
    micro_idle_log = []
    floor_hrs      = AVAILABLE_HOURS * (UTIL_TARGET_PCT / 100.0)

    all_skipped = [
        p for p in all_parts
        if should_skip(p)[0]
        and rate.get(p, 0) > 0
        and indent_monthly.get(p, 0) > 0
    ]

    machines_by_util = sorted(
        vt_machines, key=lambda m: machine_hours.get(m, 0)
    )

    for m in machines_by_util:
        remaining = round(AVAILABLE_HOURS - machine_hours.get(m, 0), 4)
        if remaining < 0.05:
            continue

        # STEP 0: 90% floor — uncapped extension
        current_util_hrs = machine_hours.get(m, 0)
        if current_util_hrs < floor_hrs:
            needed     = round(floor_hrs - current_util_hrs, 4)
            parts_on_m = [row for row in plan if row["Machine"] == m]
            if parts_on_m:
                parts_on_m_sorted = sorted(
                    parts_on_m,
                    key=lambda r: priority_scores.get(r["Part"], 0),
                    reverse=True,
                )
                for row in parts_on_m_sorted:
                    if needed <= 0.001:
                        break
                    p_ext     = row["Part"]
                    r_ext     = rate.get(p_ext, 1)
                    ext_hrs   = min(needed, remaining)
                    if ext_hrs < 0.001:
                        continue
                    extra_qty = round(ext_hrs * r_ext, 0)

                    row["Run_Hours"]      = round(
                        float(row.get("Run_Hours", 0)) + ext_hrs, 3
                    )
                    row["Production_Qty"] = round(
                        float(row.get("Production_Qty", 0)) + extra_qty, 0
                    )
                    row["Total_Hrs_Used"] = round(
                        float(row.get("Changeover_Hrs", 0))
                        + float(row["Run_Hours"]), 3
                    )
                    row["Type"] = str(row.get("Type", "Primary")) + "+Floor90"

                    machine_hours[m]         = round(
                        machine_hours.get(m, 0) + ext_hrs, 4
                    )
                    current_inventory[p_ext] = round(
                        current_inventory.get(p_ext, 0) + extra_qty, 0
                    )
                    remaining = round(remaining - ext_hrs, 4)
                    needed    = round(needed - ext_hrs, 4)
                    new_util  = round(
                        machine_hours.get(m, 0) / AVAILABLE_HOURS * 100, 1
                    )
                    print(f"    [S0-FLOOR90] {p_ext:28s} on {m:15s}  "
                          f"+{ext_hrs:.2f}h  qty+={extra_qty:.0f}  "
                          f"util now {new_util}%  [uncapped]")

        if remaining < 0.05:
            continue

        # STEP 1: Extend existing parts (OPD-capped)
        parts_on_machine = list(
            {row["Part"] for row in plan if row["Machine"] == m}
        )
        for p in sorted(
            parts_on_machine,
            key=lambda x: priority_scores.get(x, 0),
            reverse=True,
        ):
            if remaining < 0.05:
                break
            daily_p  = indent_daily.get(p, 0)
            r_val    = rate.get(p, 1)
            inv_now  = current_inventory.get(p, 0)
            headroom = max(0.0, opd_cap(scenario_id) * daily_p - inv_now)
            ext_hrs  = min(remaining, headroom / r_val if r_val > 0 else 0)
            if ext_hrs < 0.05:
                continue
            extra_qty = round(ext_hrs * r_val, 0)
            for row in plan:
                if row["Part"] == p and row["Machine"] == m:
                    row["Run_Hours"]      = round(
                        float(row.get("Run_Hours", 0)) + ext_hrs, 3
                    )
                    row["Production_Qty"] = round(
                        float(row.get("Production_Qty", 0)) + extra_qty, 0
                    )
                    row["Total_Hrs_Used"] = round(
                        float(row.get("Changeover_Hrs", 0))
                        + float(row["Run_Hours"]), 3
                    )
                    row["Type"] = str(row.get("Type", "Primary")) + "+Extended"
                    break
            machine_hours[m]     = round(machine_hours.get(m, 0) + ext_hrs, 4)
            current_inventory[p] = round(
                current_inventory.get(p, 0) + extra_qty, 0
            )
            remaining = round(remaining - ext_hrs, 4)
            print(f"    [S1-EXTEND]  {p:28s} on {m:15s}  "
                  f"+{ext_hrs:.2f}h  qty+={extra_qty:.0f}")

        if remaining < MIN_RUN_HOURS:
            continue

        # STEP 2: Unplanned compatible parts — colour-aware
        unplanned = [
            p for p in all_parts
            if p not in already_planned
            and m in vt_compat.get(p, [])
            and not should_skip(p)[0]
            and indent_monthly.get(p, 0) > 0
            and rate.get(p, 0) > 0
            and current_inventory.get(p, 0)
               < opd_cap(scenario_id) * indent_daily.get(p, 0)
        ]

        last_on_m  = machine_last_part.get(m)
        last_color = (
            part_color.get(last_on_m, "UNKNOWN") if last_on_m else "UNKNOWN"
        )

        def _co_sort_key(p):
            needs_co   = 0 if (last_on_m is None or last_on_m == p) else 1
            p_col      = part_color.get(p, "UNKNOWN")
            same_color = (
                0
                if (needs_co == 1 and p_col == last_color and p_col != "UNKNOWN")
                else 1
            )
            is_critical = 1 if current_inventory.get(p, 0) == 0 else 0
            sc          = priority_scores.get(p, 0)
            return (needs_co, same_color, -is_critical, -sc)

        unplanned.sort(key=_co_sort_key)

        for p in unplanned:
            if remaining < MIN_RUN_HOURS:
                break
            co_hrs   = _co_hrs_for(p, m, machine_last_part)
            eff_free = round(remaining - co_hrs, 4)
            if eff_free < MIN_RUN_HOURS:
                continue
            daily_p  = indent_daily.get(p, 0)
            r_val    = rate.get(p, 1)
            inv_now  = current_inventory.get(p, 0)
            cap_qty  = opd_cap(scenario_id) * daily_p
            headroom = max(0.0, cap_qty - inv_now)
            if headroom <= 0:
                continue
            shortfall = max(0.0, daily_p - inv_now)
            min_run   = max(
                MIN_RUN_HOURS,
                shortfall / r_val if r_val > 0 else MIN_RUN_HOURS,
            )
            run_hrs   = max(
                MIN_RUN_HOURS,
                min(eff_free, max(min_run, headroom / r_val if r_val > 0 else eff_free)),
            )

            p_color = part_color.get(p, "UNKNOWN")
            co_tag  = (
                "No CO" if co_hrs == 0 else (
                    f"CO+PURGE({p_color})"
                    if co_hrs > vt_changeover.get(m, DEFAULT_CHANGEOVER_HRS)
                    else f"CO({p_color})"
                )
            )
            qty       = _add_part_to_machine(
                p, m, run_hrs, co_hrs,
                machine_hours, machine_last_part, current_inventory,
                already_planned, plan, priority_scores,
                "Filler-Unplanned", "Primary",
            )
            remaining = round(remaining - co_hrs - run_hrs, 4)
            print(f"    [S2-UNPLAN]  {p:28s} → {m:15s}  {run_hrs:.2f}h  "
                  f"qty={qty:.0f}  [{co_tag}]")

        if remaining < MIN_RUN_HOURS:
            continue

        # STEP 3: Re-run planned parts (spare tool)
        planned_parts_elsewhere = [
            p for p in already_planned
            if m in vt_compat.get(p, [])
            and m not in [row["Machine"] for row in plan if row["Part"] == p]
            and tools_available.get(p, 1)
               > len({row["Machine"] for row in plan if row["Part"] == p})
        ]
        planned_parts_elsewhere.sort(key=lambda p: (
            0 if (last_on_m is None or last_on_m == p) else 1,
            0 if (
                part_color.get(p, "UNKNOWN") == last_color
                and last_color != "UNKNOWN"
            ) else 1,
            1 if current_inventory.get(p, 0) == 0 else 0,
            -priority_scores.get(p, 0),
        ))

        for p in planned_parts_elsewhere:
            if remaining < MIN_RUN_HOURS:
                break
            co_hrs   = _co_hrs_for(p, m, machine_last_part)
            eff_free = round(remaining - co_hrs, 4)
            if eff_free < MIN_RUN_HOURS:
                continue
            daily_p  = indent_daily.get(p, 0)
            r_val    = rate.get(p, 1)
            inv_now  = current_inventory.get(p, 0)
            cap_qty  = opd_cap(scenario_id) * daily_p
            headroom = max(0.0, cap_qty - inv_now)
            if headroom <= 0:
                continue
            shortfall = max(0.0, daily_p - inv_now)
            min_run   = max(
                MIN_RUN_HOURS,
                shortfall / r_val if r_val > 0 else MIN_RUN_HOURS,
            )
            run_hrs   = max(
                MIN_RUN_HOURS,
                min(eff_free, max(min_run, headroom / r_val if r_val > 0 else eff_free)),
            )

            qty = round(run_hrs * rate.get(p, 1), 0)
            machine_hours[m]     = round(
                machine_hours.get(m, 0) + co_hrs + run_hrs, 4
            )
            current_inventory[p] = round(
                current_inventory.get(p, 0) + qty, 0
            )
            machine_last_part[m] = p
            remaining            = round(remaining - co_hrs - run_hrs, 4)

            p_color      = part_color.get(p, "UNKNOWN")
            last_p       = machine_state.get(m)
            lc           = part_color.get(last_p, "UNKNOWN") if last_p else "UNKNOWN"
            has_purge_s3 = (
                last_p is not None and last_p != p
                and p_color != lc
                and p_color != "UNKNOWN" and lc != "UNKNOWN"
            )
            tools_used_now = (
                len({row["Machine"] for row in plan if row["Part"] == p}) + 1
            )

            plan.append({
                "Part":             p,
                "Color":            p_color,
                "Category":         part_category.get(p, "Stranger"),
                "Fixed_Machine":    part_fixed_machine.get(p, "—"),
                "Fixed_Used":       "N/A — enforcer",
                "Machine":          m,
                "Run_Hours":        round(run_hrs, 3),
                "Changeover_Hrs":   round(co_hrs, 3),
                "Total_Hrs_Used":   round(co_hrs + run_hrs, 3),
                "Rate_Per_Hour":    round(rate.get(p, 1), 2),
                "Production_Qty":   qty,
                "Monthly_Indent":   round(indent_monthly.get(p, 0), 0),
                "Daily_Indent":     round(daily_p, 2),
                "Today_Target":     round(today_target_qty.get(p, 0), 0),
                "Changeover":       "No" if co_hrs == 0 else "Yes",
                "Color_Purge":      "Yes" if has_purge_s3 else "No",
                "Type":             "Re-run (spare tool)",
                "Role":             f"Tool-Expansion (tool {tools_used_now})",
                "Tools_Available":  tools_available.get(p, 1),
                "Tools_Used":       tools_used_now,
                "Runner_Lock":      "No",
                "Priority_Score":   round(priority_scores.get(p, 0), 2),
                "Phase":            3,
                "Stagger_Adjusted": "No",
            })
            print(f"    [S3-RERUN]   {p:28s} → {m:15s}  {run_hrs:.2f}h  "
                  f"qty={qty:.0f}  color={p_color}  "
                  f"tool {tools_used_now}/{tools_available.get(p, 1)}")
            break

        if remaining < MIN_RUN_HOURS:
            continue

        # STEP 4: Skipped parts (last resort)
        skipped_candidates = [
            p for p in all_skipped
            if m in vt_compat.get(p, []) and rate.get(p, 0) > 0
        ]
        skipped_candidates.sort(key=lambda p: (
            0 if (last_on_m is None or last_on_m == p) else 1,
            0 if (
                part_color.get(p, "UNKNOWN") == last_color
                and last_color != "UNKNOWN"
            ) else 1,
            1 if current_inventory.get(p, 0) == 0 else 0,
            -indent_daily.get(p, 0),
        ))

        for p in skipped_candidates:
            if remaining < MIN_RUN_HOURS:
                break
            co_hrs   = _co_hrs_for(p, m, machine_last_part)
            eff_free = round(remaining - co_hrs, 4)
            if eff_free < MIN_RUN_HOURS:
                continue
            r_val   = rate.get(p, 1)
            run_hrs = max(
                MIN_RUN_HOURS,
                min(
                    eff_free,
                    indent_monthly.get(p, 0) / r_val if r_val > 0 else eff_free,
                ),
            )
            qty     = _add_part_to_machine(
                p, m, run_hrs, co_hrs,
                machine_hours, machine_last_part, current_inventory,
                already_planned, plan, priority_scores,
                "Filler-Skipped (last resort)", "Primary",
            )
            remaining = round(remaining - co_hrs - run_hrs, 4)
            print(f"    [S4-SKIPPED] {p:28s} → {m:15s}  {run_hrs:.2f}h  "
                  f"qty={qty:.0f}  color={part_color.get(p, '?')}")
            break

        # STEP 5: Log micro-idle
        final_remaining = round(AVAILABLE_HOURS - machine_hours.get(m, 0), 4)
        if final_remaining >= 0.25:
            util_final = round((1 - final_remaining / AVAILABLE_HOURS) * 100, 1)
            if util_final < UTIL_TARGET_PCT:
                micro_idle_log.append({
                    "Machine":         m,
                    "Idle_Hrs":        round(final_remaining, 3),
                    "Utilization_Pct": util_final,
                    "Note":            "All options exhausted / all parts at 5-day ceiling",
                })
                print(f"    [⚠ IDLE]     {m:15s}  "
                      f"{final_remaining:.2f}h idle ({util_final}%)")

    return micro_idle_log

# =============================================================
# SECTION 21 — OUTPUT VIEW BUILDERS
# =============================================================

def build_multi_machine_view(plan):
    if not plan:
        return pd.DataFrame()
    from collections import defaultdict
    part_rows = defaultdict(list)
    for row in plan:
        part_rows[row["Part"]].append(row)
    multi = {p: rows for p, rows in part_rows.items() if len(rows) > 1}
    if not multi:
        return pd.DataFrame()

    output_rows = []
    for part, rows in sorted(
        multi.items(),
        key=lambda x: -sum(r["Production_Qty"] for r in x[1]),
    ):
        daily     = indent_daily.get(part, 0)
        total_qty = sum(float(r["Production_Qty"]) for r in rows)
        for row in rows:
            output_rows.append({
                "Part":                   part,
                "Color":                  part_color.get(part, "UNKNOWN"),
                "Category":               part_category.get(part, "Stranger"),
                "Fixed_Machine":          part_fixed_machine.get(part, "—"),
                "Tools_Available":        tools_available.get(part, 1),
                "Machines_Used":          len(rows),
                "Machine":                row["Machine"],
                "Role":                   row.get("Role", "Primary"),
                "Run_Hours":              round(float(row["Run_Hours"]), 2),
                "Changeover_Hrs":         round(float(row.get("Changeover_Hrs", 0)), 2),
                "Color_Purge":            row.get("Color_Purge", "No"),
                "Production_Qty":         round(float(row["Production_Qty"]), 0),
                "Daily_Indent":           round(daily, 2),
                "Total_Qty_All_Machines": round(total_qty, 0),
                "Type":                   row.get("Type", "—"),
            })
        output_rows.append({
            "Part":                   f"  ↳ TOTAL — {part}",
            "Color":                  part_color.get(part, "UNKNOWN"),
            "Category":               "—",
            "Fixed_Machine":          part_fixed_machine.get(part, "—"),
            "Tools_Available":        tools_available.get(part, 1),
            "Machines_Used":          len(rows),
            "Machine":                f"{len(rows)} machines",
            "Role":                   "TOTAL",
            "Run_Hours":              round(sum(float(r["Run_Hours"]) for r in rows), 2),
            "Changeover_Hrs":         round(
                sum(float(r.get("Changeover_Hrs", 0)) for r in rows), 2
            ),
            "Color_Purge":            "—",
            "Production_Qty":         round(total_qty, 0),
            "Daily_Indent":           round(daily, 2),
            "Total_Qty_All_Machines": round(total_qty, 0),
            "Type":                   "—",
        })
        output_rows.append({k: "" for k in output_rows[-1].keys()})
    return pd.DataFrame(output_rows)


def build_production_vs_indent(plan, all_parts):
    if not plan:
        return pd.DataFrame()
    from collections import defaultdict
    part_qty      = defaultdict(float)
    part_machines = defaultdict(list)
    for row in plan:
        p = row["Part"]
        part_qty[p]      += float(row.get("Production_Qty", 0))
        part_machines[p].append(row["Machine"])

    rows = []
    for p in sorted(part_qty.keys()):
        daily     = indent_daily.get(p, 0)
        monthly   = indent_monthly.get(p, 0)
        inv_b     = inventory.get(p, 0)
        produced  = round(part_qty[p], 0)
        inv_after = round(inv_b + produced, 0)
        gap       = round(produced - daily, 0)
        gap_dir   = "OVER" if gap > 0 else ("UNDER" if gap < 0 else "MET")
        extra_days = round(gap / daily, 2) if daily > 0 and gap > 0 else 0.0
        rows.append({
            "Part":               p,
            "Color":              part_color.get(p, "UNKNOWN"),
            "Category":           part_category.get(p, "Stranger"),
            "Fixed_Machine":      part_fixed_machine.get(p, "—"),
            "Tools_Available":    tools_available.get(p, 1),
            "Machines":           ", ".join(dict.fromkeys(part_machines[p])),
            "Machines_Count":     len(set(part_machines[p])),
            "Total_Qty_Produced": produced,
            "Daily_Indent":       round(daily, 2),
            "Monthly_Indent":     round(monthly, 0),
            "Gap_vs_Daily":       gap,
            "Gap_Direction":      gap_dir,
            "Extra_Days_Stock":   extra_days,
            "Inventory_Before":   round(inv_b, 0),
            "Inventory_After":    inv_after,
            "Days_Coverage_After":round(inv_after / daily, 2) if daily > 0 else 0,
        })

    order_map = {"UNDER": 0, "MET": 1, "OVER": 2}
    df = pd.DataFrame(rows)
    if not df.empty:
        df["_sort"] = df["Gap_Direction"].map(order_map)
        df = (
            df.sort_values(["_sort", "Gap_vs_Daily"])
            .drop(columns=["_sort"])
            .reset_index(drop=True)
        )
    return df


def build_inventory_target_sheet(plan, all_parts, scenario_id):
    from collections import defaultdict
    part_produced = defaultdict(float)
    for row in plan:
        part_produced[row["Part"]] += float(row.get("Production_Qty", 0))

    rows = []
    for p in sorted(all_parts):
        daily    = indent_daily.get(p, 0)
        monthly  = indent_monthly.get(p, 0)
        inv_b    = inventory.get(p, 0)
        produced = round(part_produced.get(p, 0), 0)
        inv_after   = round(inv_b + produced, 0)
        days_before = round(inv_b     / daily, 2) if daily > 0 else 0
        days_after  = round(inv_after / daily, 2) if daily > 0 else 0
        target_qty  = round(TARGET_DAYS * daily, 0)
        gap_qty     = round(target_qty - inv_after, 0)
        gap_days    = max(0, round(gap_qty / daily, 2) if daily > 0 else 0)
        cap         = opd_cap(scenario_id)
        max_prod    = round(cap * daily, 0)

        if inv_after == 0:
            status = "CRITICAL"
        elif days_after < SAFETY_DAYS:
            status = "BELOW_SAFETY"
        elif days_after < TARGET_DAYS:
            status = "BUILDING"
        else:
            status = "AT_TARGET"

        net_gain = (cap - 1) * daily if daily > 0 else 0
        if gap_qty <= 0:
            est_days = "AT TARGET"
        elif net_gain <= 0:
            est_days = "N/A"
        else:
            est_days = str(math.ceil(gap_qty / net_gain)) + " days"

        skip, skip_reason = should_skip(p)
        rows.append({
            "Part":                   p,
            "Color":                  part_color.get(p, "UNKNOWN"),
            "Category":               part_category.get(p, "Stranger"),
            "Fixed_Machine":          part_fixed_machine.get(p, "—"),
            "Tools":                  tools_available.get(p, 1),
            "Monthly_Indent":         round(monthly, 0),
            "Daily_Indent":           round(daily, 2),
            "Target_Qty_5days":       target_qty,
            "Safety_Floor_Qty_3days": round(SAFETY_DAYS * daily, 0),
            "Inv_Before":             round(inv_b, 0),
            "Days_Coverage_Before":   days_before,
            "Produced_Today":         produced,
            "Inv_After":              inv_after,
            "Days_Coverage_After":    days_after,
            "Gap_to_Target_Qty":      max(0, gap_qty),
            "Gap_to_Target_Days":     gap_days,
            "Buffer_Status":          status,
            "OPD_Cap_Today_Days":     cap,
            "Max_Producible_Qty":     max_prod,
            "Est_Days_to_Target":     est_days,
            "Scheduled_Today":        (
                "YES" if produced > 0
                else "SKIPPED — AT TARGET" if inv_b >= target_qty
                else "NO — no capacity"
            ),
            "Skip_Reason":            skip_reason if skip else "",
        })

    status_order = {"CRITICAL": 0, "BELOW_SAFETY": 1, "BUILDING": 2, "AT_TARGET": 3}
    df = pd.DataFrame(rows)
    if not df.empty:
        df["_sort"] = df["Buffer_Status"].map(status_order)
        df = (
            df.sort_values(["_sort", "Gap_to_Target_Days"], ascending=[True, False])
            .drop(columns=["_sort"])
            .reset_index(drop=True)
        )
    return df


def compute_indent_horizon(parts):
    rows = []
    for p in parts:
        inv     = inventory.get(p, 0.0)
        monthly = indent_monthly.get(p, 0.0)
        daily   = indent_daily.get(p, 0.0)
        target  = today_target_qty.get(p, 0.0)
        r       = rate.get(p, 1.0)
        skip, skip_reason = should_skip(p)
        indent_hrs = monthly / r if r > 0 else 0.0
        days_cov   = inv / daily if daily > 0 else 0

        if inv == 0.0 and monthly > 0:
            status = "ZERO INV — FORCED"
        elif skip and inv >= TARGET_DAYS * daily:
            status = "AT 5-DAY TARGET — SKIP"
        elif skip:
            status = "SKIPPED"
        elif target > 0:
            status = "PRODUCTION NEEDED"
        elif monthly == 0:
            status = "NO INDENT"
        else:
            status = "INV SUFFICIENT"

        rows.append({
            "Part":             p,
            "Color":            part_color.get(p, "UNKNOWN"),
            "Fixed_Machine":    part_fixed_machine.get(p, "—"),
            "Tools":            tools_available.get(p, 1),
            "Monthly_Indent":   round(monthly, 0),
            "Indent_Hrs_Total": round(indent_hrs, 2),
            "Working_Days":     WORKING_DAYS,
            "Daily_Indent":     round(daily, 2),
            "Inventory_Now":    round(inv, 0),
            "Days_Coverage":    round(days_cov, 2),
            "Safety_Floor":     SAFETY_DAYS,
            "Target_Ceiling":   TARGET_DAYS,
            "Today_Target_Qty": round(target, 0),
            "Today_Target_Hrs": round(target / r if r > 0 else 0, 2),
            "Rate_Per_Hour":    round(r, 2),
            "Indent_Status":    status,
            "Skip_Reason":      skip_reason,
        })
    return pd.DataFrame(rows)


def build_terminal_status_sheet():
    rows = []
    all_terminals_known = set(terminal_status.keys())
    for terms in part_terminals.values():
        for t in terms:
            all_terminals_known.add(t)

    for t in sorted(all_terminals_known):
        inv           = terminal_status.get(t, None)
        is_available  = (inv is not None and inv > 0)
        parts_needing = [p for p, terms in part_terminals.items() if t in terms]
        parts_blocked = [p for p in parts_needing if not is_available]
        rows.append({
            "Terminal":               t,
            "Inventory":              round(inv, 0) if inv is not None else "NOT IN SHEET",
            "Available_Today":        "YES" if is_available else "NO — ZERO STOCK",
            "Parts_Requiring_Count":  len(parts_needing),
            "Parts_Requiring":        ", ".join(sorted(parts_needing)) if parts_needing else "—",
            "Parts_Blocked_Count":    len(parts_blocked),
            "Parts_Blocked":          ", ".join(sorted(parts_blocked)) if parts_blocked else "—",
            "Impact":                 (
                "BLOCKING — reschedule those parts" if parts_blocked
                else "No impact today"              if parts_needing
                else "No parts use this terminal"
            ),
        })
    return pd.DataFrame(rows)


def build_fixed_machine_audit():
    """
    Audit sheet showing every fixed machine assignment,
    whether the fixed machine was actually used today,
    and the reason if it was bypassed.
    """
    rows = []
    for part, fixed_m in sorted(part_fixed_machine.items()):
        category  = part_category.get(part, "Stranger")
        daily     = indent_daily.get(part, 0)
        inv       = inventory.get(part, 0)
        days_cov  = round(inv / daily, 2) if daily > 0 else 0

        # Find plan rows for this part
        # (plan is not accessible here — built in schedule(); we store
        # a reference below)
        rows.append({
            "Part":          part,
            "Category":      category,
            "Color":         part_color.get(part, "UNKNOWN"),
            "Fixed_Machine": fixed_m,
            "Daily_Indent":  round(daily, 2),
            "Inventory":     round(inv, 0),
            "Days_Coverage": days_cov,
            "In_Matrix":     "YES" if part in vt_compat else "NO",
            "Fixed_In_Matrix": (
                "YES" if fixed_m in vt_compat.get(part, []) else "NO"
            ),
        })
    return pd.DataFrame(rows)


SPECIALIZED_MACHINE_THRESHOLD = 3


def detect_specialized_machines(all_parts):
    machine_parts = {}
    for m in vt_machines:
        machine_parts[m] = [
            p for p in all_parts
            if m in vt_compat.get(p, [])
            and not should_skip(p)[0]
            and indent_monthly.get(p, 0) > 0
        ]
    specialized = {
        m for m, mp in machine_parts.items()
        if 0 < len(mp) <= SPECIALIZED_MACHINE_THRESHOLD
    }
    return (
        specialized,
        {m: machine_parts[m] for m in specialized},
        machine_parts,
    )

# =============================================================
# SECTION 22 — MAIN SCHEDULER
# =============================================================

def schedule(parts, label=""):
    print(f"\n{'─'*65}")
    print(f"  {label}  |  {len(parts)} parts  |  {len(vt_machines)} machines")
    print(f"  Safety floor: {SAFETY_DAYS} days  |  Target ceiling: {TARGET_DAYS} days")
    print(f"{'─'*65}")

    scenario_id, scenario_desc = classify_scenario(parts)
    print(f"  {scenario_desc}")
    print(f"  Effective OPD cap today: {opd_cap(scenario_id)} days")

    # Print fixed machine summary for today
    if part_fixed_machine:
        active_fixed = [
            p for p in part_fixed_machine
            if not should_skip(p)[0] and indent_monthly.get(p, 0) > 0
        ]
        print(f"\n  FIXED MACHINE ASSIGNMENTS ({len(active_fixed)} active today):")
        for p in sorted(active_fixed):
            fm    = part_fixed_machine[p]
            inv_p = inventory.get(p, 0)
            dal_p = indent_daily.get(p, 0)
            print(f"    {p:<30} → {fm:<25}  "
                  f"inv={inv_p:.0f}  daily={dal_p:.2f}  "
                  f"in_matrix={'YES' if fm in vt_compat.get(p,[]) else '⚠ NO'}")

    horizon_df = compute_indent_horizon(parts)

    active_parts = [
        p for p in parts
        if not should_skip(p)[0] and indent_monthly.get(p, 0) > 0
    ]

    priority_scores, score_rows = compute_priority_scores(active_parts)
    score_df = pd.DataFrame(score_rows) if score_rows else pd.DataFrame()

    machine_hours     = {m: 0.0 for m in vt_machines}
    machine_last_part = {m: machine_state.get(m) for m in vt_machines}
    current_inventory = inventory.copy()
    plan              = []
    already_planned   = set()
    not_planned       = []
    deferred          = []
    displaced_log     = []

    specialized_machines, spec_machine_parts, _ = detect_specialized_machines(
        list(parts)
    )
    print(f"\n  SPECIALIZED MACHINES (≤{SPECIALIZED_MACHINE_THRESHOLD} parts):")
    if specialized_machines:
        for m in sorted(
            specialized_machines,
            key=lambda m: machine_part_count.get(m, 99),
        ):
            mparts = spec_machine_parts.get(m, [])
            cnt    = machine_part_count.get(m, "?")
            print(f"    {m:<25} Part_Count={cnt:<4} parts: {', '.join(mparts)}")
    else:
        print(f"    None — all machines have >{SPECIALIZED_MACHINE_THRESHOLD} compatible parts")

    zero_inv_rr = [
        p for p in active_parts
        if current_inventory.get(p, 0) == 0
        and part_category.get(p, "Stranger") in ("Runner", "Repeater")
        and vt_compat.get(p)
    ]
    if zero_inv_rr:
        print(f"\n  V9 DISPLACEMENT PRE-PASS  "
              f"({len(zero_inv_rr)} zero-inv Runner/Repeater parts)")
    else:
        print(f"\n  V9 DISPLACEMENT PRE-PASS  "
              f"— no zero-inv Runner/Repeater parts today  ✓")

    sorted_active = sorted(
        active_parts,
        key=lambda p: priority_scores.get(p, 0),
        reverse=True,
    )

    print(f"\n  PRIMARY SCHEDULING PASS  ({len(sorted_active)} active parts)")
    print(f"  {'Part':<30} {'Color':<10} {'Fixed':<20} {'Score':>6} {'Days':>5} "
          f"{'Status':<15} {'Machine(s)':<25} {'Run':>5} {'Qty':>8}")
    print(f"  {'─'*125}")

    for part in sorted_active:
        inv_now  = current_inventory.get(part, 0)
        daily    = indent_daily.get(part, 0)
        monthly  = indent_monthly.get(part, 0)
        score    = priority_scores.get(part, 0)
        tools    = tools_available.get(part, 1)
        category = part_category.get(part, "Stranger")
        color    = part_color.get(part, "UNKNOWN")
        fixed_m  = part_fixed_machine.get(part, "—")
        days_cov = inv_now / daily if daily > 0 else 999

        buf_label = (
            "CRITICAL"     if inv_now == 0 else
            "BELOW_SAFETY" if days_cov < SAFETY_DAYS else
            "BUILDING"     if days_cov < TARGET_DAYS else
            "AT_TARGET"
        )

        if monthly == 0:
            deferred.append({
                "Part": part, "Color": color, "Fixed_Machine": fixed_m,
                "Category": category, "Reason": "Monthly indent = 0",
            })
            print(f"  {part:<30} {color:<10} {fixed_m:<20} {score:>6.1f} {days_cov:>5.1f} "
                  f"{'DEFERRED':<15}  —  no indent")
            continue

        t_blocked, t_reason = terminal_blocked(part)
        if t_blocked:
            not_planned.append({
                "Part":                part,
                "Color":               color,
                "Fixed_Machine":       fixed_m,
                "Category":            category,
                "Score":               score,
                "Tools":               tools,
                "Days_Coverage":       round(days_cov, 2),
                "Buffer_Status":       buf_label,
                "Daily_Indent":        round(daily, 2),
                "Inventory_Now":       round(inv_now, 0),
                "Compatible_Machines": ", ".join(vt_compat.get(part, [])),
                "Reason":              t_reason,
                "Action_Needed":       "Replenish terminal stock or reschedule",
                "Terminal_Required":   ", ".join(part_terminals.get(part, [])),
            })
            print(f"  {part:<30} {color:<10} {fixed_m:<20} {score:>6.1f} {days_cov:>5.1f} "
                  f"{buf_label:<15}  ✗ TERMINAL ZERO STOCK")
            continue

        if not vt_compat.get(part):
            not_planned.append({
                "Part": part, "Color": color, "Fixed_Machine": fixed_m,
                "Category": category, "Score": score,
                "Daily_Indent": round(daily, 2),
                "Inventory_Now": round(inv_now, 0), "Tools": tools,
                "Compatible_Machines": "NONE DEFINED",
                "Reason": "Not in VT_Matrix",
                "Action_Needed": "Add to VT_Matrix",
            })
            print(f"  {part:<30} {color:<10} {fixed_m:<20} {score:>6.1f} {days_cov:>5.1f} "
                  f"{buf_label:<15}  ✗ NOT IN MATRIX")
            continue

        new_rows = assign_part(
            part, scenario_id, machine_hours, machine_last_part,
            current_inventory, plan, already_planned, priority_scores,
        )

        if new_rows:
            plan.extend(new_rows)
            machines_used = [r["Machine"] for r in new_rows]
            total_qty     = sum(float(r["Production_Qty"]) for r in new_rows)
            total_run     = sum(float(r["Run_Hours"]) for r in new_rows)
            machines_str  = ", ".join(machines_used)
            flag  = " [MULTI]"    if len(new_rows) > 1 else ""
            flag += " [ZERO-INV]" if inv_now == 0    else ""
            purges = sum(1 for r in new_rows if r.get("Color_Purge") == "Yes")
            flag  += f" [PURGE×{purges}]" if purges > 0 else ""
            # Annotate if fixed machine was used or bypassed
            fixed_tags = set(r.get("Fixed_Used", "") for r in new_rows)
            if "YES" in fixed_tags:
                flag += " [FIXED ✓]"
            elif "FALLBACK" in fixed_tags:
                flag += " [FIXED FALLBACK]"
            print(f"  {part:<30} {color:<10} {fixed_m:<20} {score:>6.1f} {days_cov:>5.1f} "
                  f"{buf_label:<15}  {machines_str:<25}  "
                  f"{total_run:>5.2f}  {total_qty:>8.0f}  ✓{flag}")
        else:
            not_planned.append({
                "Part":                part,
                "Color":               color,
                "Fixed_Machine":       fixed_m,
                "Category":            category,
                "Score":               score,
                "Tools":               tools,
                "Days_Coverage":       round(days_cov, 2),
                "Buffer_Status":       buf_label,
                "Daily_Indent":        round(daily, 2),
                "Inventory_Now":       round(inv_now, 0),
                "Compatible_Machines": ", ".join(vt_compat.get(part, [])),
                "Reason":              "No compatible machine has capacity",
                "Action_Needed":       "Review matrix or add machines",
                "Terminal_Required":   ", ".join(part_terminals.get(part, [])) or "—",
            })
            print(f"  {part:<30} {color:<10} {fixed_m:<20} {score:>6.1f} {days_cov:>5.1f} "
                  f"{buf_label:<15}  ✗ NO CAPACITY")

    # Post-primary displacement (V9 original)
    unscheduled_zero_rr = [
        p for p in zero_inv_rr if p not in already_planned
    ]
    if unscheduled_zero_rr:
        print(f"\n  V9 DISPLACEMENT PASS  "
              f"({len(unscheduled_zero_rr)} zero-inv R/R unscheduled)")
        for part in unscheduled_zero_rr:
            success = displace_for_zero_inv(
                part, machine_hours, machine_last_part,
                current_inventory, plan, already_planned, priority_scores,
            )
            if success:
                displaced_log.append(part)
                not_planned[:] = [
                    r for r in not_planned if r.get("Part") != part
                ]
            else:
                print(f"      ↳ DISPLACEMENT FAILED  {part}  "
                      f"— no suitable victim found")
    else:
        print(f"\n  V9 DISPLACEMENT PASS  "
              f"— no unscheduled zero-inv R/R parts  ✓")

    # Runner priority enforcement
    runner_priority_log = enforce_runner_priority(
        plan, machine_hours, machine_last_part,
        current_inventory, already_planned, priority_scores,
    )
    newly_planned = {
        r["Runner_Part"]
        for r in runner_priority_log
        if "PLANNED" in r.get("Result", "")
    }
    if newly_planned:
        not_planned[:] = [
            r for r in not_planned
            if r.get("Part") not in newly_planned
        ]

    # Utilisation enforcer
    micro_idle = utilization_enforcer(
        plan, machine_hours, machine_last_part,
        list(parts), already_planned,
        current_inventory, scenario_id, priority_scores,
    )

    # CO stagger
    print(f"\n  CO Quantity Stagger  "
          f"(min gap = {MIN_CO_GAP_HRS * 60:.0f} min after CO finishes)")
    n_adj = stagger_co_by_quantity(
        plan, vt_machines, scenario_id, current_inventory
    )
    print(f"    {'No adjustments needed  ✓' if n_adj == 0 else str(n_adj) + ' adjustment(s) made'}")

    stagger_changeovers(plan, vt_machines)

    # Build output views
    multi_machine_df   = build_multi_machine_view(plan)
    prod_vs_indent_df  = build_production_vs_indent(plan, list(parts))
    inv_target_df      = build_inventory_target_sheet(plan, list(parts), scenario_id)
    runner_priority_df = (
        pd.DataFrame(runner_priority_log) if runner_priority_log else pd.DataFrame()
    )

    # Fixed machine adherence summary for output
    fixed_adherence_rows = []
    for part, fixed_m in sorted(part_fixed_machine.items()):
        part_plan_rows = [r for r in plan if r["Part"] == part]
        if not part_plan_rows:
            status_fa = "NOT PLANNED TODAY"
            machine_used = "—"
        else:
            machines_used_set = {r["Machine"] for r in part_plan_rows}
            fixed_used_today  = fixed_m in machines_used_set
            other_machines    = machines_used_set - {fixed_m}
            machine_used      = ", ".join(sorted(machines_used_set))
            if fixed_used_today and not other_machines:
                status_fa = "FIXED MACHINE USED"
            elif fixed_used_today and other_machines:
                status_fa = "FIXED + FALLBACK USED"
            else:
                status_fa = "FALLBACK ONLY — fixed had no capacity"

        fixed_adherence_rows.append({
            "Part":            part,
            "Category":        part_category.get(part, "Stranger"),
            "Color":           part_color.get(part, "UNKNOWN"),
            "Fixed_Machine":   fixed_m,
            "Fixed_In_Matrix": "YES" if fixed_m in vt_compat.get(part, []) else "NO",
            "Machine_Used":    machine_used,
            "Status":          status_fa,
            "Daily_Indent":    round(indent_daily.get(part, 0), 2),
            "Inventory":       round(inventory.get(part, 0), 0),
        })
    fixed_adherence_df = pd.DataFrame(fixed_adherence_rows)

    # Daily indent status
    indent_status_rows = []
    for row in plan:
        p       = row["Part"]
        planned = float(row.get("Production_Qty") or 0)
        daily   = indent_daily.get(p, 0)
        inv_b   = inventory.get(p, 0)
        meets   = planned >= daily
        indent_status_rows.append({
            "Part":               p,
            "Color":              part_color.get(p, "UNKNOWN"),
            "Category":           part_category.get(p, "Stranger"),
            "Fixed_Machine":      part_fixed_machine.get(p, "—"),
            "Machine":            row.get("Machine", "—"),
            "Color_Purge":        row.get("Color_Purge", "No"),
            "Role":               row.get("Role", "Primary"),
            "Priority_Score":     round(row.get("Priority_Score", 0), 2),
            "Buffer_Status":      (
                "CRITICAL"     if inv_b == 0 else
                "BELOW_SAFETY" if (daily > 0 and inv_b / daily < SAFETY_DAYS)
                else "BUILDING" if (daily > 0 and inv_b / daily < TARGET_DAYS)
                else "AT_TARGET"
            ),
            "Run_Hours":          round(float(row.get("Run_Hours") or 0), 2),
            "Planned_Qty":        round(planned, 0),
            "Daily_Indent":       round(daily, 2),
            "Gap_vs_Daily":       round(daily - planned, 0),
            "Meets_Daily_Indent": "YES ✓" if meets else "NO ✗",
            "Inventory_Before":   round(inv_b, 0),
            "Total_Available":    round(inv_b + planned, 0),
            "Covers_With_Inv":    "YES ✓" if (inv_b + planned) >= daily else "NO ✗",
        })
    indent_status_df = pd.DataFrame(indent_status_rows)
    if not indent_status_df.empty:
        indent_status_df = indent_status_df.sort_values(
            ["Meets_Daily_Indent", "Gap_vs_Daily"],
            ascending=[True, False],
        ).reset_index(drop=True)

    # Inventory health
    inv_rows = []
    for p in parts:
        inv_b    = inventory.get(p, 0)
        daily    = indent_daily.get(p, 0)
        produced = sum(float(r["Production_Qty"]) for r in plan if r["Part"] == p)
        inv_after = inv_b + produced
        days_cov  = inv_after / daily if daily > 0 else 0
        inv_rows.append({
            "Part":            p,
            "Color":           part_color.get(p, "UNKNOWN"),
            "Fixed_Machine":   part_fixed_machine.get(p, "—"),
            "Tools":           tools_available.get(p, 1),
            "Rate_Per_Hour":   round(rate.get(p, 0), 2),
            "Monthly_Indent":  round(indent_monthly.get(p, 0), 0),
            "Daily_Indent":    round(daily, 2),
            "Inv_Before":      round(inv_b, 0),
            "Produced_Today":  round(produced, 0),
            "Inv_After_Today": round(inv_after, 0),
            "Days_Coverage":   round(days_cov, 2),
            "Safety_Floor":    SAFETY_DAYS,
            "Target_Ceiling":  TARGET_DAYS,
            "Status":          (
                "AT_TARGET" if days_cov >= TARGET_DAYS else
                "OK"        if days_cov >= SAFETY_DAYS else
                "LOW"       if days_cov >= 1 else
                "CRITICAL"
            ),
        })

    # Machine utilisation
    mach_rows = []
    for m in vt_machines:
        used      = machine_hours.get(m, 0)
        parts_run = list({r["Part"] for r in plan if r["Machine"] == m})
        co_count  = sum(
            1 for r in plan if r["Machine"] == m and r.get("Changeover") == "Yes"
        )
        purge_count = sum(
            1 for r in plan if r["Machine"] == m and r.get("Color_Purge") == "Yes"
        )
        colors_on_machine = list({
            part_color.get(r["Part"], "UNKNOWN") for r in plan if r["Machine"] == m
        })
        # Which fixed parts ran on this machine?
        fixed_parts_on_m = [
            p for p in parts_run
            if part_fixed_machine.get(p) == m
        ]
        util_pct = round(used / AVAILABLE_HOURS * 100, 1)
        mach_rows.append({
            "Machine":              m,
            "Fixed_Parts":          ", ".join(fixed_parts_on_m) if fixed_parts_on_m else "—",
            "Colors_Today":         ", ".join(sorted(colors_on_machine)),
            "Color_Purges":         purge_count,
            "Used_Hours":           round(used, 2),
            "Unused_Hours":         round(AVAILABLE_HOURS - used, 2),
            "Utilization_%":        util_pct,
            "Status":               (
                "FULL"      if used >= AVAILABLE_HOURS - 0.3 else
                "GOOD"      if used >= AVAILABLE_HOURS * 0.98 else
                "OK"        if used >= AVAILABLE_HOURS * 0.90 else
                "PARTIAL"   if used >= AVAILABLE_HOURS * 0.85 else
                "UNDERUSED"
            ),
            "Parts_Planned":        len(parts_run),
            "Changeovers":          co_count,
            "Last_Part_Run":        machine_last_part.get(m) or "—",
            "All_Parts_On_Machine": ", ".join(parts_run) if parts_run else "— idle —",
        })

    micro_df           = pd.DataFrame(micro_idle)   if micro_idle   else pd.DataFrame()
    plan_df            = pd.DataFrame(plan)         if plan         else pd.DataFrame()
    def_df             = pd.DataFrame(deferred)     if deferred     else pd.DataFrame()
    not_df             = pd.DataFrame(not_planned)  if not_planned  else pd.DataFrame()
    mach_df            = pd.DataFrame(mach_rows)
    inv_df             = pd.DataFrame(inv_rows)

    if not plan_df.empty:
        plan_df.insert(0, "Planning_Date",  str(PLANNING_DATE))
        plan_df.insert(1, "Scenario",       scenario_desc)
        plan_df.insert(2, "Working_Days",   WORKING_DAYS)
        plan_df.insert(3, "Safety_Days",    SAFETY_DAYS)
        plan_df.insert(4, "Target_Days",    TARGET_DAYS)
        plan_df.insert(5, "OPD_Cap_Today",  opd_cap(scenario_id))

    n_runner_placed = sum(
        1 for r in runner_priority_log if "PLANNED" in r.get("Result", "")
    )
    n_runner_failed = sum(
        1 for r in runner_priority_log if "FAILED" in r.get("Result", "")
    )

    # Fixed machine adherence summary in console
    if fixed_adherence_rows:
        used_fixed  = sum(1 for r in fixed_adherence_rows if "FIXED MACHINE USED" in r["Status"])
        fallback    = sum(1 for r in fixed_adherence_rows if "FALLBACK" in r["Status"])
        not_planned_fixed = sum(1 for r in fixed_adherence_rows if r["Status"] == "NOT PLANNED TODAY")
        print(f"\n  FIXED MACHINE ADHERENCE:")
        print(f"    Fixed machine used correctly : {used_fixed}")
        print(f"    Fallback used (fixed full)   : {fallback}")
        print(f"    Not planned today            : {not_planned_fixed}")

    print(f"\n  {'='*65}")
    print(f"  SCHEDULE SUMMARY — {scenario_desc}")
    print(f"    Parts planned              : {len(already_planned)}")
    print(f"    Runner priority placed     : {n_runner_placed}")
    print(f"    Runner priority failed     : {n_runner_failed}")
    print(f"    Displaced (zero-inv R/R)   : {len(displaced_log)}")
    print(f"    Not planned                : {len(not_planned)}")
    print(f"    Deferred                   : {len(deferred)}")
    if not mach_df.empty:
        print(f"    Avg utilization            : {mach_df['Utilization_%'].mean():.1f}%")
        print(f"    Total colour purges        : {mach_df['Color_Purges'].sum()}")
    if not inv_target_df.empty:
        at_t = (inv_target_df["Buffer_Status"] == "AT_TARGET").sum()
        bld  = (inv_target_df["Buffer_Status"] == "BUILDING").sum()
        bls  = (inv_target_df["Buffer_Status"] == "BELOW_SAFETY").sum()
        crt  = (inv_target_df["Buffer_Status"] == "CRITICAL").sum()
        print(f"\n    Inventory target status (after today):")
        print(f"      AT_TARGET   (≥5 days) : {at_t}")
        print(f"      BUILDING  (3–5 days)  : {bld}")
        print(f"      BELOW_SAFETY (<3 days): {bls}")
        print(f"      CRITICAL  (0 pcs)     : {crt}")
    print(f"  {'='*65}")

    return (
        plan_df, def_df, not_df, mach_df, inv_df,
        machine_last_part, horizon_df, indent_status_df,
        score_df, micro_df, multi_machine_df, prod_vs_indent_df,
        inv_target_df, runner_priority_df, fixed_adherence_df,
    )

# =============================================================
# SECTION 23 — PART AUDIT
# =============================================================

vt_parts         = data_valid[
    data_valid["Material"].isin(vt_matrix["Part"])
]["Material"].unique()
all_vt_parts_raw = list(data["Material"].unique())
matrix_parts     = set(str(p).strip() for p in vt_matrix["Part"] if pd.notna(p))
zero_rate_set    = set(data_zero_rate["Material"].unique())

audit_rows = []
for part in all_vt_parts_raw:
    inv      = inventory.get(part, 0.0)
    r_val    = rate.get(part, None)
    monthly  = indent_monthly.get(part, 0.0)
    daily    = indent_daily.get(part, 0.0)
    row_data = data[data["Material"] == part]
    ct_raw   = row_data[vt_col_cycletime].values[0] if len(row_data) else "—"
    cv_raw   = row_data[vt_col_cavity].values[0]    if len(row_data) else "—"
    tools    = tools_available.get(part, 1)
    color    = part_color.get(part, "UNKNOWN")
    fixed_m  = part_fixed_machine.get(part, "—")
    days_cov = inv / daily if daily > 0 else 0

    part_terms    = part_terminals.get(part, [])
    t_blk, t_rsn  = terminal_blocked(part)
    terms_str      = ", ".join(part_terms) if part_terms else "—"
    terms_zero     = (
        ", ".join(sorted([
            f"{t}(inv={terminal_status.get(t, 0):.0f})"
            for t in part_terms if terminal_status.get(t, 0) <= 0
        ])) or "—"
    )

    if part in zero_rate_set or r_val is None:
        status, gate, reason = "ZERO/MISSING CYCLE TIME", "GATE 1", f"Cycle time={ct_raw}"
    elif part not in matrix_parts:
        status, gate, reason = "NOT IN VT_MATRIX", "GATE 2", "No compatible machine"
    elif monthly == 0:
        status, gate, reason = "ZERO/MISSING INDENT", "GATE 3", "Monthly indent = 0"
    elif daily <= MIN_DAILY_INDENT:
        status, gate, reason = "SKIPPED (LOW INDENT)", "GATE 4a", f"Daily ≤ {MIN_DAILY_INDENT}"
    elif (monthly / r_val if r_val else 0) <= MIN_INDENT_HOURS:
        status, gate, reason = "SKIPPED (TRIVIAL RUN)", "GATE 4b", f"Monthly hrs ≤ {MIN_INDENT_HOURS}h"
    elif daily > 0 and inv >= TARGET_DAYS * daily:
        status, gate, reason = (
            f"AT {TARGET_DAYS}-DAY TARGET — SKIP TODAY", "GATE 5",
            f"Inv ({inv:.0f}) ≥ {TARGET_DAYS}×daily ({TARGET_DAYS * daily:.0f})",
        )
    elif t_blk:
        status, gate, reason = "BLOCKED — TERMINAL ZERO STOCK", "GATE 6", t_rsn
    else:
        status, gate, reason = "ENTERS SCHEDULER", "—", "Passed all gates"

    audit_rows.append({
        "Part":               part,
        "Color":              color,
        "Fixed_Machine":      fixed_m,
        "Gate_Failed":        gate,
        "Reason":             reason,
        "Terminals_Required": terms_str,
        "Terminals_Zero_Inv": terms_zero,
        "Monthly_Indent":     round(monthly, 0),
        "Daily_Indent":       round(daily, 2),
        "Inventory":          round(inv, 0),
        "Days_Coverage":      round(days_cov, 2),
        "Safety_Floor":       SAFETY_DAYS,
        "Target_Ceiling":     TARGET_DAYS,
        "Tools":              tools,
        "Rate_Per_Hour":      round(r_val, 2) if r_val else "—",
        "Cycle_Time":         ct_raw,
        "Cavity":             cv_raw,
        "Status":             status,
    })

audit_df    = pd.DataFrame(audit_rows)
gate_counts = audit_df["Status"].value_counts()
print(f"\n  Part audit ({len(all_vt_parts_raw)} total):")
for status, count in gate_counts.items():
    marker = "✓" if status == "ENTERS SCHEDULER" else "·"
    print(f"    {marker}  {status:<52}: {count:>4}")

vt_terminal_status_df = build_terminal_status_sheet()

if not vt_terminal_status_df.empty:
    zero_rows           = vt_terminal_status_df[
        vt_terminal_status_df["Available_Today"] == "NO — ZERO STOCK"
    ]
    total_blocked_today = zero_rows["Parts_Blocked_Count"].sum()
    print(f"\n  Terminal Status Summary:")
    print(f"    Terminals tracked         : {len(vt_terminal_status_df)}")
    print(f"    Terminals at zero stock   : "
          f"{(vt_terminal_status_df['Available_Today']=='NO — ZERO STOCK').sum()}")
    print(f"    Parts blocked today       : {int(total_blocked_today)}")
    if not zero_rows.empty:
        for _, row in zero_rows.iterrows():
            print(f"      ✗  {row['Terminal']:<20} inv={row['Inventory']}  "
                  f"blocks: {row['Parts_Blocked']}")
else:
    print(f"\n  Terminal Status: No terminal data loaded — constraint inactive.")

# =============================================================
# SECTION 24 — RUN
# =============================================================

(vt_plan, vt_def, vt_not, vt_mach, vt_inv,
 vt_state, vt_horizon, vt_indent_status,
 vt_scores, vt_micro, vt_multi_machine,
 vt_prod_vs_indent, vt_inv_target,
 vt_runner_priority, vt_fixed_adherence) = schedule(vt_parts, "VT Machines")

save_machine_state(vt_state)

# =============================================================
# SECTION 25 — MACHINE-WISE PLAN
# =============================================================

def build_machine_wise_plan(plan_df):
    if plan_df.empty:
        return pd.DataFrame()

    def sf(val, default=0.0):
        try:
            v = float(val)
            return v if not np.isnan(v) else default
        except (TypeError, ValueError):
            return default

    rows = []
    for m in vt_machines:
        machine_rows = plan_df[plan_df["Machine"] == m].copy()
        if machine_rows.empty:
            continue

        for _, pr in machine_rows.iterrows():
            p     = pr.get("Part", "—")
            co_h  = sf(pr.get("Changeover_Hrs", 0))
            run_h = sf(pr.get("Run_Hours", 0))
            r_val = sf(pr.get("Rate_Per_Hour", 0))
            rows.append({
                "Machine":           m,
                "Part":              p,
                "Color":             part_color.get(p, "UNKNOWN"),
                "Category":          part_category.get(p, "Stranger"),
                "Fixed_Machine":     part_fixed_machine.get(p, "—"),
                "Fixed_Used":        pr.get("Fixed_Used", "N/A"),
                "Role":              pr.get("Role", "Primary"),
                "Tools_Available":   pr.get("Tools_Available", 1),
                "Priority_Score":    round(sf(pr.get("Priority_Score", 0)), 2),
                "Rate_Per_Hour":     round(r_val, 2),
                "Run_Hours":         round(run_h, 2),
                "Changeover_Hrs":    round(co_h, 3),
                "Changeover_Needed": pr.get("Changeover", "No") or "No",
                "Color_Purge":       pr.get("Color_Purge", "No") or "No",
                "Production_Qty":    round(sf(pr.get("Production_Qty", 0)), 0),
                "Daily_Indent":      round(sf(pr.get("Daily_Indent", 0)), 2),
                "Today_Target":      round(sf(pr.get("Today_Target", 0)), 0),
                "Monthly_Indent":    round(sf(pr.get("Monthly_Indent", 0)), 0),
                "Type":              pr.get("Type", "Primary") or "Primary",
                "Row_Type":          "Part",
            })

        co_total    = machine_rows["Changeover_Hrs"].apply(lambda x: sf(x, 0)).sum()
        run_total   = machine_rows["Run_Hours"].apply(lambda x: sf(x, 0)).sum()
        qty_total   = machine_rows["Production_Qty"].apply(lambda x: sf(x, 0)).sum()
        co_count    = int(machine_rows["Changeover"].eq("Yes").sum())
        purge_count = (
            int(machine_rows["Color_Purge"].eq("Yes").sum())
            if "Color_Purge" in machine_rows.columns else 0
        )
        hrs_total   = round(co_total + run_total, 2)
        colors_list = sorted({part_color.get(p, "UNKNOWN") for p in machine_rows["Part"]})
        fixed_on_m  = [
            p for p in machine_rows["Part"]
            if part_fixed_machine.get(p) == m
        ]

        rows.append({
            "Machine":           m,
            "Part":              f"TOTAL — {m}",
            "Color":             ", ".join(colors_list),
            "Category":          "—",
            "Fixed_Machine":     ", ".join(fixed_on_m) if fixed_on_m else "—",
            "Fixed_Used":        "—",
            "Role":              "—",
            "Tools_Available":   "—",
            "Priority_Score":    "—",
            "Rate_Per_Hour":     "—",
            "Run_Hours":         round(run_total, 2),
            "Changeover_Hrs":    round(co_total, 2),
            "Changeover_Needed": f"{co_count} changeover(s)",
            "Color_Purge":       f"{purge_count} purge(s)",
            "Production_Qty":    round(qty_total, 0),
            "Daily_Indent":      "—",
            "Today_Target":      "—",
            "Monthly_Indent":    "—",
            "Type":              (
                f"Total {hrs_total}h / {AVAILABLE_HOURS}h  |  "
                f"Idle {round(AVAILABLE_HOURS - hrs_total, 2)}h  |  "
                f"Util {round(hrs_total / AVAILABLE_HOURS * 100, 1)}%"
            ),
            "Row_Type":          "Summary",
        })
        rows.append({k: "" for k in rows[-1].keys()})

    return pd.DataFrame(rows)


def build_co_queue(plan, machines):
    events = _collect_co_events(plan, machines)
    if not events:
        return pd.DataFrame()

    events.sort(key=lambda e: e["natural_start"])
    rows                 = []
    tool_changer_free_at = 0.0

    for pos, ev in enumerate(events, 1):
        natural_start        = _recompute_natural_start(ev, plan)
        co_h                 = ev["co_duration"]
        actual_start         = max(natural_start, tool_changer_free_at)
        wait_min             = round((actual_start - natural_start) * 60, 1)
        tool_changer_free_at = actual_start + co_h

        before_color = part_color.get(ev["part_before"], "UNKNOWN")
        after_color  = part_color.get(ev["part_after"],  "UNKNOWN")
        color_change = (
            before_color != after_color
            and before_color != "UNKNOWN"
            and after_color  != "UNKNOWN"
        )

        note = ""
        if wait_min > 0:
            note = (
                f"Machine may need to wait ~{wait_min}min. "
                f"Continue running previous part until team arrives."
            )
        if color_change:
            note = (note + "  " if note else "") + (
                f"COLOUR CHANGE: {before_color} → {after_color}. "
                f"10-min purge required."
            )

        rows.append({
            "Queue_Position":  pos,
            "Machine":         ev["machine"],
            "Part_Before":     ev["part_before"],
            "Color_Before":    before_color,
            "Part_After":      ev["part_after"],
            "Color_After":     after_color,
            "Color_Change":    "YES — PURGE" if color_change else "No",
            "CO_Duration_Min": round(co_h * 60, 1),
            "Note":            (
                note if note
                else "Tool changer available immediately. No colour change."
            ),
        })

    return pd.DataFrame(rows)

# =============================================================
# SECTION 26 — EXCEL OUTPUT
# =============================================================

from openpyxl import load_workbook
from openpyxl.styles import PatternFill, Font, Alignment
from openpyxl.utils import get_column_letter

HEADER_COLORS = {
    "VT_Plan_By_Machine":       "0D6E6E",
    "VT_CO_Queue":              "375623",
    "VT_Plan":                  "1F4E79",
    "VT_Daily_Indent_Status":   "0F4C2A",
    "VT_Multi_Machine_Parts":   "4A235A",
    "VT_Production_vs_Indent":  "154360",
    "VT_Inventory_Target":      "1B4F72",
    "VT_Priority_Scores":       "2C4770",
    "VT_Machine_Util":          "375623",
    "VT_Not_Planned":           "7B2C2C",
    "VT_Deferred":              "7F6000",
    "VT_Inventory_Health":      "4A235A",
    "VT_Indent_Horizon":        "154360",
    "VT_Part_Audit":            "1C3557",
    "VT_Micro_Idle":            "5C3D2E",
    "VT_Terminal_Status":       "7B1C1C",
    "VT_Runner_Priority_Log":   "7B3F00",
    "VT_Fixed_Adherence":       "1A5276",   # deep blue — fixed machine audit
}

STATUS_FILLS = {
    "FULL":         PatternFill("solid", fgColor="C6EFCE"),
    "GOOD":         PatternFill("solid", fgColor="DDEBF7"),
    "OK":           PatternFill("solid", fgColor="EBF5E1"),
    "PARTIAL":      PatternFill("solid", fgColor="FFEB9C"),
    "UNDERUSED":    PatternFill("solid", fgColor="FFC7CE"),
    "AT_TARGET":    PatternFill("solid", fgColor="C6EFCE"),
    "BUILDING":     PatternFill("solid", fgColor="DDEBF7"),
    "BELOW_SAFETY": PatternFill("solid", fgColor="FFEB9C"),
    "CRITICAL":     PatternFill("solid", fgColor="FFC7CE"),
    "LOW":          PatternFill("solid", fgColor="FFEB9C"),
    "YES ✓":        PatternFill("solid", fgColor="C6EFCE"),
    "NO ✗":         PatternFill("solid", fgColor="FFC7CE"),
    "OVER":         PatternFill("solid", fgColor="DDEBF7"),
    "UNDER":        PatternFill("solid", fgColor="FFC7CE"),
    "MET":          PatternFill("solid", fgColor="C6EFCE"),
    "YES — PURGE":  PatternFill("solid", fgColor="FFC7CE"),
    "Yes":          PatternFill("solid", fgColor="FFEB9C"),
    "PRODUCTION NEEDED":                         PatternFill("solid", fgColor="FFC7CE"),
    "ZERO INV — FORCED":                         PatternFill("solid", fgColor="FFD7D7"),
    "INV SUFFICIENT":                            PatternFill("solid", fgColor="C6EFCE"),
    "SKIPPED":                                   PatternFill("solid", fgColor="EDEDED"),
    f"AT {TARGET_DAYS}-DAY TARGET — SKIP TODAY": PatternFill("solid", fgColor="C6EFCE"),
    "ZERO/MISSING CYCLE TIME":                   PatternFill("solid", fgColor="FFC7CE"),
    "NOT IN VT_MATRIX":                          PatternFill("solid", fgColor="FFEB9C"),
    "ZERO/MISSING INDENT":                       PatternFill("solid", fgColor="FFEB9C"),
    "ENTERS SCHEDULER":                          PatternFill("solid", fgColor="C6EFCE"),
    "YES":                                       PatternFill("solid", fgColor="C6EFCE"),
    "NO — ZERO STOCK":                           PatternFill("solid", fgColor="FFC7CE"),
    "BLOCKING — reschedule those parts":         PatternFill("solid", fgColor="FFC7CE"),
    "No impact today":                           PatternFill("solid", fgColor="C6EFCE"),
    "BLOCKED — TERMINAL ZERO STOCK":             PatternFill("solid", fgColor="FFC7CE"),
    "PLANNED — displacement successful":         PatternFill("solid", fgColor="C6EFCE"),
    "PLANNED — partial (machine hours limited)": PatternFill("solid", fgColor="FFEB9C"),
    "PLANNED — free capacity":                   PatternFill("solid", fgColor="DDEBF7"),
    "FAILED — insufficient capacity on all compatible machines":
                                                 PatternFill("solid", fgColor="FFC7CE"),
    "FAILED — safety check after carve":         PatternFill("solid", fgColor="FFC7CE"),
    # Fixed adherence statuses
    "FIXED MACHINE USED":                        PatternFill("solid", fgColor="C6EFCE"),
    "FIXED + FALLBACK USED":                     PatternFill("solid", fgColor="DDEBF7"),
    "FALLBACK ONLY — fixed had no capacity":     PatternFill("solid", fgColor="FFEB9C"),
    "NOT PLANNED TODAY":                         PatternFill("solid", fgColor="EDEDED"),
}


def style_sheet(ws, header_hex):
    for cell in ws[1]:
        cell.fill      = PatternFill("solid", fgColor=header_hex)
        cell.font      = Font(bold=True, color="FFFFFF", size=11)
        cell.alignment = Alignment(
            horizontal="center", vertical="center", wrap_text=True
        )
    ws.row_dimensions[1].height = 32
    for col in ws.columns:
        col_letter = get_column_letter(col[0].column)
        max_len    = max(
            (len(str(c.value)) for c in col if c.value is not None), default=8
        )
        ws.column_dimensions[col_letter].width = max(10, min(55, max_len + 3))
    headers = [cell.value for cell in ws[1]]
    status_cols = [
        "Status", "Indent_Status", "Meets_Daily", "Covers_With",
        "Gap_Direction", "Buffer_Status", "Scheduled_Today",
        "Color_Change", "Color_Purge", "Available_Today", "Impact",
        "Result", "Fixed_Used",
    ]
    for col_idx, col_name in enumerate(headers, start=1):
        if col_name and any(x in str(col_name) for x in status_cols):
            for row in ws.iter_rows(
                min_row=2, min_col=col_idx, max_col=col_idx
            ):
                for cell in row:
                    fill = STATUS_FILLS.get(str(cell.value))
                    if fill:
                        cell.fill = fill
    ws.freeze_panes = "A2"


def style_fixed_adherence_sheet(ws):
    """Style the VT_Fixed_Adherence sheet."""
    style_sheet(ws, HEADER_COLORS["VT_Fixed_Adherence"])
    headers    = [c.value for c in ws[1]]
    status_col = headers.index("Status") + 1 if "Status" in headers else None
    if not status_col:
        return
    for row in ws.iter_rows(min_row=2):
        status_val = str(row[status_col - 1].value)
        fill       = STATUS_FILLS.get(status_val)
        if fill:
            for cell in row:
                if cell.fill.fill_type == "none" or cell.fill.fgColor.rgb in (
                    "00000000", "FFFFFFFF"
                ):
                    cell.fill = fill
        if status_val == "FALLBACK ONLY — fixed had no capacity":
            for cell in row:
                cell.font = Font(bold=True)


def style_runner_priority_sheet(ws):
    style_sheet(ws, HEADER_COLORS["VT_Runner_Priority_Log"])
    headers    = [c.value for c in ws[1]]
    result_col = headers.index("Result") + 1 if "Result" in headers else None
    if not result_col:
        return
    for row in ws.iter_rows(min_row=2):
        result_val = str(row[result_col - 1].value)
        if "FAILED" in result_val:
            for cell in row:
                cell.font = Font(bold=True, color="7B1C1C")
        elif "PLANNED — displacement" in result_val:
            for cell in row:
                cell.font = Font(bold=True)


def style_inv_target_sheet(ws):
    style_sheet(ws, HEADER_COLORS["VT_Inventory_Target"])
    headers    = [c.value for c in ws[1]]
    status_col = (
        headers.index("Buffer_Status") + 1 if "Buffer_Status" in headers else None
    )
    row_fills = {
        "CRITICAL":     PatternFill("solid", fgColor="FFD7D7"),
        "BELOW_SAFETY": PatternFill("solid", fgColor="FFF2CC"),
        "BUILDING":     PatternFill("solid", fgColor="DDEEFF"),
        "AT_TARGET":    PatternFill("solid", fgColor="E2EFDA"),
    }
    for row in ws.iter_rows(min_row=2):
        if not status_col:
            continue
        status = str(row[status_col - 1].value)
        fill   = row_fills.get(status)
        if fill:
            for cell in row:
                if cell.fill.fill_type == "none" or cell.fill.fgColor.rgb in (
                    "00000000", "FFFFFFFF"
                ):
                    cell.fill = fill
        if status == "CRITICAL":
            for cell in row:
                cell.font = Font(bold=True)


def style_machine_wise_sheet(ws):
    header_fill  = PatternFill("solid", fgColor="1F4E79")
    summary_fill = PatternFill("solid", fgColor="0D9488")
    part_fills   = [
        PatternFill("solid", fgColor="EFF6FF"),
        PatternFill("solid", fgColor="F0FDF4"),
    ]
    co_fill      = PatternFill("solid", fgColor="FEF9C3")
    purge_fill   = PatternFill("solid", fgColor="FFC7CE")
    fixed_fill   = PatternFill("solid", fgColor="E8F4FD")   # light blue for fixed rows

    for cell in ws[1]:
        cell.fill      = header_fill
        cell.font      = Font(bold=True, color="FFFFFF", size=11)
        cell.alignment = Alignment(
            horizontal="center", vertical="center", wrap_text=True
        )
    ws.row_dimensions[1].height = 30

    headers      = [cell.value for cell in ws[1]]
    row_type_col = headers.index("Row_Type")     + 1 if "Row_Type"     in headers else None
    co_col       = headers.index("Changeover_Needed") + 1 if "Changeover_Needed" in headers else None
    purge_col    = headers.index("Color_Purge")  + 1 if "Color_Purge"  in headers else None
    machine_col  = headers.index("Machine")      + 1 if "Machine"      in headers else None
    fixed_u_col  = headers.index("Fixed_Used")   + 1 if "Fixed_Used"   in headers else None

    machine_color_idx = 0
    current_machine   = None
    for row in ws.iter_rows(min_row=2):
        row_type = row[row_type_col - 1].value if row_type_col else ""
        machine  = row[machine_col  - 1].value if machine_col  else ""
        if machine and machine != current_machine:
            current_machine   = machine
            machine_color_idx = (machine_color_idx + 1) % 2
        if row_type == "Summary":
            for cell in row:
                cell.fill = summary_fill
                cell.font = Font(bold=True, color="FFFFFF", size=11)
                cell.alignment = Alignment(horizontal="center", vertical="center")
        elif row_type == "Part":
            base_fill = part_fills[machine_color_idx]
            for cell in row:
                cell.fill      = base_fill
                cell.alignment = Alignment(vertical="center")
            # Highlight fixed machine rows
            if fixed_u_col and str(row[fixed_u_col - 1].value) == "YES":
                for cell in row:
                    cell.fill = fixed_fill
            if co_col and str(row[co_col - 1].value) == "Yes":
                row[co_col - 1].fill = co_fill
            if purge_col and str(row[purge_col - 1].value) == "Yes":
                row[purge_col - 1].fill = purge_fill

    for col in ws.columns:
        col_letter = get_column_letter(col[0].column)
        max_len    = max((len(str(c.value)) for c in col if c.value), default=8)
        ws.column_dimensions[col_letter].width = max(10, min(45, max_len + 3))
    ws.freeze_panes = "B2"


def style_prod_vs_indent_sheet(ws):
    style_sheet(ws, HEADER_COLORS["VT_Production_vs_Indent"])
    headers  = [c.value for c in ws[1]]
    gap_col  = headers.index("Gap_Direction") + 1 if "Gap_Direction" in headers else None
    over_fill  = PatternFill("solid", fgColor="DDEBF7")
    under_fill = PatternFill("solid", fgColor="FFC7CE")
    met_fill   = PatternFill("solid", fgColor="C6EFCE")
    for row in ws.iter_rows(min_row=2):
        if gap_col:
            cell  = row[gap_col - 1]
            value = str(cell.value)
            if value == "OVER":
                cell.fill = over_fill
            elif value == "UNDER":
                cell.fill = under_fill
                for c in row:
                    c.font = Font(bold=True)
            elif value == "MET":
                cell.fill = met_fill


def style_multi_machine_sheet(ws):
    style_sheet(ws, HEADER_COLORS["VT_Multi_Machine_Parts"])
    headers  = [c.value for c in ws[1]]
    role_col = headers.index("Role") + 1 if "Role" in headers else None
    total_fill   = PatternFill("solid", fgColor="0D9488")
    primary_fill = PatternFill("solid", fgColor="EFF6FF")
    expand_fill  = PatternFill("solid", fgColor="FEF9C3")
    for row in ws.iter_rows(min_row=2):
        if not role_col:
            continue
        role = str(row[role_col - 1].value)
        if role == "TOTAL":
            for cell in row:
                cell.fill = total_fill
                cell.font = Font(bold=True, color="FFFFFF", size=11)
        elif "Tool-Expansion" in role:
            for cell in row:
                cell.fill = expand_fill
        elif role == "Primary":
            for cell in row:
                cell.fill = primary_fill


def style_co_queue_sheet(ws):
    style_sheet(ws, HEADER_COLORS["VT_CO_Queue"])
    headers    = [c.value for c in ws[1]]
    cc_col     = headers.index("Color_Change") + 1 if "Color_Change" in headers else None
    purge_fill = PatternFill("solid", fgColor="FFC7CE")
    for row in ws.iter_rows(min_row=2):
        if cc_col and str(row[cc_col - 1].value).startswith("YES"):
            for cell in row:
                if cell.fill.fill_type == "none" or cell.fill.fgColor.rgb in (
                    "00000000", "FFFFFFFF"
                ):
                    cell.fill = purge_fill
            row[cc_col - 1].font = Font(bold=True, color="7B1C1C")


# Write Excel
print(f"\nWriting output → {output_path}")

vt_mw   = build_machine_wise_plan(vt_plan)
vt_co_q = build_co_queue(
    [{k: v for k, v in r.items()} for r in vt_plan.to_dict("records")]
    if not vt_plan.empty else [],
    vt_machines,
)

sheets = {
    "VT_Plan_By_Machine":       vt_mw,
    "VT_CO_Queue":              vt_co_q,
    "VT_Plan":                  vt_plan,
    "VT_Runner_Priority_Log":   vt_runner_priority,
    "VT_Fixed_Adherence":       vt_fixed_adherence,     # ← NEW
    "VT_Inventory_Target":      vt_inv_target,
    "VT_Multi_Machine_Parts":   vt_multi_machine,
    "VT_Production_vs_Indent":  vt_prod_vs_indent,
    "VT_Daily_Indent_Status":   vt_indent_status,
    "VT_Priority_Scores":       vt_scores,
    "VT_Machine_Util":          vt_mach,
    "VT_Not_Planned":           vt_not,
    "VT_Deferred":              vt_def,
    "VT_Inventory_Health":      vt_inv,
    "VT_Indent_Horizon":        vt_horizon,
    "VT_Part_Audit":            audit_df,
    "VT_Terminal_Status":       vt_terminal_status_df,
}
if not vt_micro.empty:
    sheets["VT_Micro_Idle"] = vt_micro

with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    for sheet_name, df in sheets.items():
        if df is not None and not df.empty:
            df.to_excel(writer, sheet_name=sheet_name, index=False)

wb = load_workbook(output_path)

if "VT_Plan_By_Machine"      in wb.sheetnames: style_machine_wise_sheet(wb["VT_Plan_By_Machine"])
if "VT_Production_vs_Indent" in wb.sheetnames: style_prod_vs_indent_sheet(wb["VT_Production_vs_Indent"])
if "VT_Multi_Machine_Parts"  in wb.sheetnames: style_multi_machine_sheet(wb["VT_Multi_Machine_Parts"])
if "VT_Inventory_Target"     in wb.sheetnames: style_inv_target_sheet(wb["VT_Inventory_Target"])
if "VT_CO_Queue"             in wb.sheetnames: style_co_queue_sheet(wb["VT_CO_Queue"])
if "VT_Runner_Priority_Log"  in wb.sheetnames: style_runner_priority_sheet(wb["VT_Runner_Priority_Log"])
if "VT_Fixed_Adherence"      in wb.sheetnames: style_fixed_adherence_sheet(wb["VT_Fixed_Adherence"])

for sheet_name, header_hex in HEADER_COLORS.items():
    if sheet_name in wb.sheetnames and sheet_name not in (
        "VT_Plan_By_Machine", "VT_Production_vs_Indent",
        "VT_Multi_Machine_Parts", "VT_Inventory_Target",
        "VT_CO_Queue", "VT_Runner_Priority_Log", "VT_Fixed_Adherence",
    ):
        style_sheet(wb[sheet_name], header_hex)

if "VT_Daily_Indent_Status" in wb.sheetnames:
    ws_is      = wb["VT_Daily_Indent_Status"]
    headers_is = [c.value for c in ws_is[1]]
    meets_col  = (
        headers_is.index("Meets_Daily_Indent") + 1
        if "Meets_Daily_Indent" in headers_is else None
    )
    covers_col = (
        headers_is.index("Covers_With_Inv") + 1
        if "Covers_With_Inv" in headers_is else None
    )
    for row in ws_is.iter_rows(min_row=2):
        if meets_col:
            cell = row[meets_col - 1]
            cell.fill = (
                PatternFill("solid", fgColor="C6EFCE")
                if str(cell.value) == "YES ✓"
                else PatternFill("solid", fgColor="FFC7CE")
            )
        if covers_col:
            cell = row[covers_col - 1]
            cell.fill = (
                PatternFill("solid", fgColor="C6EFCE")
                if str(cell.value) == "YES ✓"
                else PatternFill("solid", fgColor="FFEB9C")
            )
        if meets_col and str(row[meets_col - 1].value) == "NO ✗":
            for cell in row:
                cell.font = Font(bold=True)

for name, color in HEADER_COLORS.items():
    if name in wb.sheetnames:
        wb[name].sheet_properties.tabColor = color

wb.save(output_path)
print(f"  Formatting applied  ✓")

# =============================================================
# SECTION 27 — FINAL SUMMARY
# =============================================================

print(f"\n{'='*65}")
print(f"  Smart APS V9 + Fixed Machine + Runner Priority  —  {PLANNING_DATE}")
print(f"  Safety floor: {SAFETY_DAYS} days  |  Target ceiling: {TARGET_DAYS} days")
print(f"  Colour purge penalty : {int(COLOR_PURGE_HRS * 60)} min")
print(f"{'='*65}")

for status, count in audit_df["Status"].value_counts().items():
    marker = "✓" if status == "ENTERS SCHEDULER" else "·"
    print(f"  {marker} {status:<52}: {count:>4}")

print(f"\n  Results:")
print(f"    Plan rows written      : {len(vt_plan):>4}")
print(f"    Not planned            : {len(vt_not):>4}")
print(f"    Deferred               : {len(vt_def):>4}")

if not vt_fixed_adherence.empty:
    fa_used     = (vt_fixed_adherence["Status"] == "FIXED MACHINE USED").sum()
    fa_fallback = vt_fixed_adherence["Status"].str.contains("FALLBACK", na=False).sum()
    fa_unplan   = (vt_fixed_adherence["Status"] == "NOT PLANNED TODAY").sum()
    print(f"\n  Fixed Machine Adherence ({len(vt_fixed_adherence)} fixed parts):")
    print(f"    Fixed machine used     : {fa_used}")
    print(f"    Fallback used          : {fa_fallback}")
    print(f"    Not planned today      : {fa_unplan}")

if not vt_runner_priority.empty:
    rp_placed = (vt_runner_priority["Result"].str.contains("PLANNED", na=False)).sum()
    rp_failed = (vt_runner_priority["Result"].str.contains("FAILED",  na=False)).sum()
    print(f"\n  Runner Priority Enforcement:")
    print(f"    Runners placed via priority : {rp_placed}")
    print(f"    Runners unresolvable        : {rp_failed}")
    for _, r in vt_runner_priority[
        vt_runner_priority["Result"].str.contains("PLANNED", na=False)
    ].iterrows():
        print(f"      ✓  {r['Runner_Part']:<28}  → {r['Machine_Assigned']}  "
              f"qty={r['Qty_Produced']:.0f}  "
              f"fixed={r.get('Fixed_Used','N/A')}  "
              f"displaced={r['Displacement_Used']}")
    for _, r in vt_runner_priority[
        vt_runner_priority["Result"].str.contains("FAILED", na=False)
    ].iterrows():
        print(f"      ✗  {r['Runner_Part']:<28}  {r['Result']}")

if not vt_inv_target.empty:
    at_t = (vt_inv_target["Buffer_Status"] == "AT_TARGET").sum()
    bld  = (vt_inv_target["Buffer_Status"] == "BUILDING").sum()
    bls  = (vt_inv_target["Buffer_Status"] == "BELOW_SAFETY").sum()
    crt  = (vt_inv_target["Buffer_Status"] == "CRITICAL").sum()
    print(f"\n  Inventory target status (end of day):")
    print(f"    AT_TARGET   (≥5 days) : {at_t:>4}")
    print(f"    BUILDING  (3–5 days)  : {bld:>4}")
    print(f"    BELOW_SAFETY (<3 days): {bls:>4}")
    print(f"    CRITICAL  (0 pcs)     : {crt:>4}")

if not vt_mach.empty:
    print(f"\n  Machine utilization:")
    print(f"    Average         : {vt_mach['Utilization_%'].mean():.1f}%")
    print(f"    UNDERUSED       : {(vt_mach['Status'] == 'UNDERUSED').sum()} machines")
    print(f"    Total purges    : {vt_mach['Color_Purges'].sum()} colour changeovers")

if not vt_terminal_status_df.empty:
    zero_today    = (vt_terminal_status_df["Available_Today"] == "NO — ZERO STOCK").sum()
    blocked_today = int(vt_terminal_status_df["Parts_Blocked_Count"].sum())
    print(f"\n  Terminal constraint:")
    print(f"    Terminals at zero stock : {zero_today}")
    print(f"    Parts blocked           : {blocked_today}")

print(f"\n  Output → {output_path}")
print(f"  State  → {MACHINE_STATE_FILE}")
print(f"\n  UPDATE DAILY: PLANNING_DATE = date(2026, 4, 10)")
print(f"{'='*65}")